# UASTHN-2 vs UASTHN-1

**Prompt Explaining the Algorithm**

>in this plot we examine results of an costumized version of the algorithm called UASTHN.
>
>STHN is an older algorith which predicts geo-localization location of a thermal image on the bigger satellite image in 2 stages (homography). first stage (IHN1 module) predicts a coarse 4 corners points, and the second (IHN2 module) refine it and predicts the final 4 corners.
>
>IHN is an iterative algorithm that calculates from center of satellite image in iter 0, in n steps (lets say 6) to final piont.
>
>UASTHN, calcualtes an uncertainity estimation (ue) for each prediction by crop augmenting (lets say 4 crops + 1 original thermal) from thermal image and feeding them with one satellite image to IHN1, then calculates STD of 5 outputs as UE. then with one threshold we decide it ue greater than thresh to reject the predicted original homography, else we accept and go to IHN2 to refine it (if using two stages STHN).
>
>we have costumize it to have 2 thresholds, one lower thresh and one higher thresh. after first ue estimation in UASTHN, if ue smaller than lower thresh we reject, else if greater than high thresh we accept and go to next stage, else if ue between the bounds (suspecious) we do more crop augmentation (say 4 more and we'll have 4+4+1 predictions of IHN1) and feed them to IHN1 to calculate UE2. then we aggregate UE with UE2 to have UE_final and we compare this with a mid thresh for acc or rej. lets call 2 thresh as UASTHN-2 and 1 thresh as UASTHN-1 for now.
>
>i have run UASTHN-2 and gathered their ue ue2, four corner preds of final stage and all stages. we formulate this as a classification problem by defining another thresh for Center Error (CE thresh like 100 pixel); Trues are preds center less than ce thresh, and others are falses. Positive when we accept the pred, and else it is Negative.

In [ ]:
import os
import math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# sss CONFIG
# ==============================================================================
PROJECT_ROOT = os.getcwd()
PRED_EXCEL  = "js_datasets/Heydar/crop5_9-two-bing.xlsx"
REF_EXCEL   = "js_datasets/Heydar/crop5-two-bing.xlsx"
GT_EXCEL    = "js_datasets/Heydar/gt.xlsx"

USE_REFERENCE    = True
USE_GROUND_TRUTH = True

SHOW_PRED = True
SHOW_REF  = True
SHOW_GT   = True

# ── Algorithm display names (rename freely) ───────────────────────────────────
NAME_1T = "UASTHN-1"   # single-threshold classification (was 1-Thresh)
NAME_2T = "UASTHN-2"   # two-step three-zone classification (was 2-Thresh)
NAME_STHN = "STHN"     # underlying base model (no UE-based filtering)

# ── Master switch: when False, skip everything related to UASTHN-2 / sus / ue_fin
USE_UASTHN2 = False

# ── PRED UE thresholds ────────────────────────────────────────────────────────
PRED_UE1_THRESH = 7    # UASTHN-1: accept if ue < this value
PRED_UE_LOW     = 4    # UASTHN-2 zone low  : ue < UE_LOW  → Accepted
PRED_UE_HIGH    = 10   # UASTHN-2 zone high : ue > UE_HIGH → Rejected
PRED_UE2_MID    = 7    # UASTHN-2: for suspicious images, accept if ue_fin < UE2_MID

# ── REF UE thresholds (independent from PRED) ────────────────────────────────
REF_UE1_THRESH = 10
REF_UE_LOW     = 7
REF_UE_HIGH    = 7
REF_UE2_MID    = 7

# ── Spatial error threshold (shared by PRED and REF) ─────────────────────────
CE_THRESH = 100   # pixels

NUM_IMAGES     = 50
GRID_COLS      = 6
ALPHA_BLEND    = 1

RANDOM_SELECTION = True
SEED             = 42

SAVE_FIGURE = False
SAVE_PATH   = "js_excels/overlay_grid.png"
FIG_DPI     = 180

# Sub-Heatmap Area
UE_RANGE   = (0, 10)
DIST_RANGE = (0, 110)
DISP_RANGE = (0, 110)
HEAT_BINS  = 15

# ==============================================================================
# sss HELPERS — paths, file naming
# ==============================================================================
def resolve_img_path(path_text):
    path_text = str(path_text)
    if os.path.exists(path_text):
        return path_text
    candidate = os.path.join(PROJECT_ROOT, path_text)
    if os.path.exists(candidate):
        return candidate
    return None


def get_filename_from_path(path_text):
    return os.path.splitext(os.path.basename(str(path_text)))[0]


# ==============================================================================
# sss HELPERS — ue_fin computation
# ==============================================================================
def build_ue_fin(df, ue_low, ue_high):
    """
    Compute ue_fin column:
      - If ue is in suspicious range [ue_low, ue_high] → use ue2 (fallback to ue if ue2 is NaN)
      - Otherwise → ue_fin = ue
    This is the single value used for all UASTHN-2 decisions and plots.
    Suspicious-zone membership is ALWAYS determined from the ORIGINAL ue value,
    so "the same suspicious data" can be tracked before (ue) and after (ue_fin).
    """
    if 'ue' not in df.columns:
        return pd.Series([np.nan] * len(df), index=df.index)

    ue  = df['ue'].copy()
    ue2 = df['ue2'].copy() if 'ue2' in df.columns else ue.copy()

    sus_mask = (ue >= ue_low) & (ue <= ue_high)

    ue_fin = ue.copy()
    ue_fin[sus_mask] = ue2[sus_mask].fillna(ue[sus_mask])

    return ue_fin


# ==============================================================================
# sss HELPERS — drawing (quads, iteration path, borders)
# ==============================================================================
def draw_quad(image, row, color, thickness, label=None,
              x1_col='x1', y1_col='y1', x2_col='x2', y2_col='y2',
              x3_col='x3', y3_col='y3', x4_col='x4', y4_col='y4'):
    quad = np.array(
        [[row[x1_col], row[y1_col]],
         [row[x2_col], row[y2_col]],
         [row[x4_col], row[y4_col]],
         [row[x3_col], row[y3_col]]],
        dtype=np.int32,
    )
    cv2.polylines(image, [quad], isClosed=True, color=color, thickness=thickness)
    if label is not None:
        center = np.mean(quad, axis=0).astype(np.int32)
        cv2.putText(image, label,
                    (int(center[0]) - 30, int(center[1])),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)


def draw_iteration_path(image, row, img_h, img_w):
    """
    Draw iteration path: image_center → iter1..6 centers → final pred center
    Dashed red line + colored dots.
    """
    ITER_COLS = [
        ('x1_1','y1_1','x2_1','y2_1','x3_1','y3_1','x4_1','y4_1'),
        ('x1_2','y1_2','x2_2','y2_2','x3_2','y3_2','x4_2','y4_2'),
        ('x1_3','y1_3','x2_3','y2_3','x3_3','y3_3','x4_3','y4_3'),
        ('x1_4','y1_4','x2_4','y2_4','x3_4','y3_4','x4_4','y4_4'),
        ('x1_5','y1_5','x2_5','y2_5','x3_5','y3_5','x4_5','y4_5'),
        ('x1_6','y1_6','x2_6','y2_6','x3_6','y3_6','x4_6','y4_6'),
    ]
    RED    = (255, 60, 60)
    ORANGE = (255, 165, 0)
    DOT_R  = max(4, img_w // 120)
    LINE_T = max(1, img_w // 400)

    waypoints = [(img_w // 2, img_h // 2)]  # step 0: image center

    for cols in ITER_COLS:
        x1c, y1c, x2c, y2c, x3c, y3c, x4c, y4c = cols
        if all(c in row.index for c in cols) and not any(pd.isna(row[c]) for c in cols):
            cx = int(round((row[x1c] + row[x2c] + row[x3c] + row[x4c]) / 4))
            cy = int(round((row[y1c] + row[y2c] + row[y3c] + row[y4c]) / 4))
            waypoints.append((cx, cy))

    if all(c in row.index for c in ['x1','y1','x2','y2','x3','y3','x4','y4']):
        if not any(pd.isna(row[c]) for c in ['x1','y1','x2','y2','x3','y3','x4','y4']):
            cx = int(round((row['x1'] + row['x2'] + row['x3'] + row['x4']) / 4))
            cy = int(round((row['y1'] + row['y2'] + row['y3'] + row['y4']) / 4))
            waypoints.append((cx, cy))

    if len(waypoints) < 2:
        return

    dash_len = max(8, img_w // 80)
    gap_len  = max(4, img_w // 160)

    def draw_dashed_line(img, pt1, pt2, color, thickness, dash, gap):
        x1, y1 = pt1; x2, y2 = pt2
        dx, dy = x2 - x1, y2 - y1
        dist = math.hypot(dx, dy)
        if dist < 1:
            return
        ux, uy = dx / dist, dy / dist
        pos, drawing = 0.0, True
        while pos < dist:
            end_pos = min(pos + (dash if drawing else gap), dist)
            if drawing:
                cv2.line(img,
                         (int(x1 + ux * pos),     int(y1 + uy * pos)),
                         (int(x1 + ux * end_pos),  int(y1 + uy * end_pos)),
                         color, thickness, cv2.LINE_AA)
            pos, drawing = end_pos, not drawing

    for i in range(len(waypoints) - 1):
        draw_dashed_line(image, waypoints[i], waypoints[i+1], RED, LINE_T, dash_len, gap_len)

    for step_i, (px, py) in enumerate(waypoints):
        is_final = (step_i == len(waypoints) - 1)
        is_start = (step_i == 0)
        color  = (255, 255, 255) if is_start else (ORANGE if is_final else RED)
        radius = DOT_R + 2 if (is_final or is_start) else DOT_R
        cv2.circle(image, (px, py), radius,     (0, 0, 0), -1)
        cv2.circle(image, (px, py), radius - 1, color,     -1)
        if not is_start and not is_final:
            cv2.putText(image, str(step_i),
                        (px + radius + 2, py - radius),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 150), 1, cv2.LINE_AA)
        elif is_final:
            cv2.putText(image, "F",
                        (px + radius + 2, py - radius),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 220, 100), 1, cv2.LINE_AA)


def classify_ue_2t_with_zone(ue_val, ue_fin_val, ue_low, ue_high, ue2_mid):
    """
    Full classification keeping zone information for border coloring.
    Returns: 'accepted' | 'rejected' | 'suspicious_acc' | 'suspicious_rej'
    """
    if pd.isna(ue_val):
        return 'accepted'
    was_suspicious = (ue_val >= ue_low) and (ue_val <= ue_high)
    if not was_suspicious:
        return 'accepted' if ue_val < ue_low else 'rejected'
    else:
        fin = ue_fin_val if not pd.isna(ue_fin_val) else ue_val
        return 'suspicious_acc' if fin < ue2_mid else 'suspicious_rej'


def add_border(image, ue_class, border_width=28):
    result = image.copy()
    bw     = border_width
    GREEN  = (0, 200, 0)
    RED    = (210, 0, 0)
    YELLOW = (220, 200, 0)

    if ue_class == 'accepted':
        tl = br = GREEN
    elif ue_class == 'rejected':
        tl = br = RED
    elif ue_class == 'suspicious_acc':
        tl, br = YELLOW, GREEN
    elif ue_class == 'suspicious_rej':
        tl, br = YELLOW, RED
    else:
        tl = br = YELLOW

    result[:bw,  :]  = tl   # top
    result[:,  :bw]  = tl   # left
    result[-bw:, :]  = br   # bottom
    result[:,  -bw:] = br   # right
    return result


# ==============================================================================
# sss HELPERS — geometry: corners, centers, distances
# ==============================================================================
def calculate_distance(p1, p2):
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


def get_corner_points(row, suffix=''):
    return np.array([[row[f"x1{suffix}"], row[f"y1{suffix}"]],
                     [row[f"x2{suffix}"], row[f"y2{suffix}"]],
                     [row[f"x3{suffix}"], row[f"y3{suffix}"]],
                     [row[f"x4{suffix}"], row[f"y4{suffix}"]]])


def get_center_point(row, suffix=''):
    return np.mean(get_corner_points(row, suffix), axis=0)


def compute_distances(row1, row2):
    c1, c2   = get_center_point(row1), get_center_point(row2)
    co1, co2 = get_corner_points(row1), get_corner_points(row2)
    corner_dists = [calculate_distance(co1[i], co2[i]) for i in range(4)]
    return {
        'center_dist':     calculate_distance(c1, c2),
        'avg_corner_dist': np.mean(corner_dists),
        'max_corner_dist': np.max(corner_dists),
        'min_corner_dist': np.min(corner_dists),
    }


def compute_iter_dist_disp(row, img_h, img_w):
    """
    dist  = distance from iteration-0 (image center) to iteration-6 center
    disp  = total path displacement: sum of segment lengths from
            iter0 -> iter1 -> ... -> iter6 (NOT including the final refined box)
    Returns (dist, disp) or (np.nan, np.nan) if iteration columns are missing.
    """
    ITER_COLS = [
        ('x1_1','y1_1','x2_1','y2_1','x3_1','y3_1','x4_1','y4_1'),
        ('x1_2','y1_2','x2_2','y2_2','x3_2','y3_2','x4_2','y4_2'),
        ('x1_3','y1_3','x2_3','y2_3','x3_3','y3_3','x4_3','y4_3'),
        ('x1_4','y1_4','x2_4','y2_4','x3_4','y3_4','x4_4','y4_4'),
        ('x1_5','y1_5','x2_5','y2_5','x3_5','y3_5','x4_5','y4_5'),
        ('x1_6','y1_6','x2_6','y2_6','x3_6','y3_6','x4_6','y4_6'),
    ]
    centers = [np.array([img_w / 2.0, img_h / 2.0])]
    for cols in ITER_COLS:
        if all(c in row.index for c in cols) and not any(pd.isna(row[c]) for c in cols):
            cx = (row[cols[0]] + row[cols[2]] + row[cols[4]] + row[cols[6]]) / 4.0
            cy = (row[cols[1]] + row[cols[3]] + row[cols[5]] + row[cols[7]]) / 4.0
            centers.append(np.array([cx, cy]))

    if len(centers) < 2:
        return np.nan, np.nan

    dist = calculate_distance(centers[0], centers[-1])
    disp = sum(calculate_distance(centers[i], centers[i+1]) for i in range(len(centers) - 1))
    return dist, disp


# ==============================================================================
# sss HELPERS — classification / confusion-matrix metrics
# ==============================================================================
def compute_final_accept_2t(df, ue_low, ue_high, ue2_mid):
    """
    UASTHN-2 accept/reject using ue_fin (already built in df).
      Non-suspicious rows: decided directly by ue vs ue_low / ue_high.
      Suspicious rows:     decided by ue_fin vs ue2_mid.
    """
    if 'ue' not in df.columns:
        return pd.Series([True] * len(df), index=df.index)

    ue     = df['ue']
    ue_fin = df['ue_fin']

    result   = pd.Series(False, index=df.index)
    sus_mask = (ue >= ue_low) & (ue <= ue_high)

    result[~sus_mask & (ue < ue_low)]  = True
    result[~sus_mask & (ue > ue_high)] = False
    result[sus_mask] = ue_fin[sus_mask] < ue2_mid

    return result.astype(bool)


def cm_metrics(acc_pred, acc_real):
    tp = int(np.sum( acc_pred &  acc_real))
    tn = int(np.sum(~acc_pred & ~acc_real))
    fp = int(np.sum( acc_pred & ~acc_real))
    fn = int(np.sum(~acc_pred &  acc_real))
    total = tp + tn + fp + fn
    accuracy      = (tp + tn) / total * 100 if total > 0 else 0.0
    pos_precision = tp / (tp + fp) * 100 if (tp + fp) > 0 else 0.0
    pos_recall    = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0.0
    neg_precision = tn / (tn + fn) * 100 if (tn + fn) > 0 else 0.0
    neg_recall    = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0.0
    f1_pos = 2*pos_precision*pos_recall/(pos_precision+pos_recall) if (pos_precision+pos_recall) > 0 else 0.0
    f1_neg = 2*neg_precision*neg_recall/(neg_precision+neg_recall) if (neg_precision+neg_recall) > 0 else 0.0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, total=total,
                accuracy=accuracy,
                pos_precision=pos_precision, pos_recall=pos_recall, f1_pos=f1_pos,
                neg_precision=neg_precision, neg_recall=neg_recall, f1_neg=f1_neg)


# ==============================================================================
# sss HELPERS — distribution stats for a numeric series
# ==============================================================================
def series_stats(vals):
    """Return dict of basic descriptive stats; handles empty arrays gracefully."""
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return dict(n=0, mean=np.nan, median=np.nan, std=np.nan, min=np.nan, max=np.nan)
    return dict(n=len(vals), mean=vals.mean(), median=np.median(vals),
                std=vals.std(), min=vals.min(), max=vals.max())


# ==============================================================================
# sss LOAD EXCEL FILES
# ==============================================================================
pred_excel_path = PRED_EXCEL if os.path.exists(PRED_EXCEL) else os.path.join(PROJECT_ROOT, PRED_EXCEL)
if not os.path.exists(pred_excel_path):
    raise FileNotFoundError(f"Prediction Excel not found: {pred_excel_path}")

pred_df = pd.read_excel(pred_excel_path)
ref_df  = None
gt_df   = None

if USE_REFERENCE:
    ref_excel_path = REF_EXCEL if os.path.exists(REF_EXCEL) else os.path.join(PROJECT_ROOT, REF_EXCEL)
    if os.path.exists(ref_excel_path):
        ref_df = pd.read_excel(ref_excel_path)

gt_excel_path = GT_EXCEL if os.path.exists(GT_EXCEL) else os.path.join(PROJECT_ROOT, GT_EXCEL)
if USE_GROUND_TRUTH and os.path.exists(gt_excel_path):
    gt_df = pd.read_excel(gt_excel_path)

required_cols = ["x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4", "sat", "th"]
missing = [c for c in required_cols if c not in pred_df.columns]
if missing:
    raise KeyError(f"Missing columns in prediction Excel: {missing}")

PRED_NAME = get_filename_from_path(PRED_EXCEL)
REF_NAME  = get_filename_from_path(REF_EXCEL)
GT_NAME   = get_filename_from_path(GT_EXCEL)

pred_has_ue  = "ue"  in pred_df.columns
pred_has_ue2 = "ue2" in pred_df.columns
ref_has_ue   = ref_df is not None and "ue"  in ref_df.columns
ref_has_ue2  = ref_df is not None and "ue2" in ref_df.columns


# ==============================================================================
# sss UE2 FALLBACK — ensure ue2 exists; fill missing values from ue
# ==============================================================================
if pred_has_ue:
    if not pred_has_ue2:
        pred_df['ue2'] = pred_df['ue'].copy()
        pred_has_ue2   = True
    else:
        pred_df['ue2'] = pred_df['ue2'].fillna(pred_df['ue'])

if ref_has_ue:
    if not ref_has_ue2:
        ref_df['ue2'] = ref_df['ue'].copy()
        ref_has_ue2   = True
    else:
        ref_df['ue2'] = ref_df['ue2'].fillna(ref_df['ue'])


# ==============================================================================
# sss UE_FIN COMPUTATION (only meaningful when USE_UASTHN2)
# ==============================================================================
# ue_fin[i] = ue2[i]  if ue[i] in suspicious range [UE_LOW, UE_HIGH]
#           = ue[i]   otherwise
# Suspicious-zone membership ALWAYS derives from the original ue, so the same
# rows can be tracked "before" (via ue) and "after" (via ue_fin).
if pred_has_ue:
    if USE_UASTHN2:
        pred_df['ue_fin'] = build_ue_fin(pred_df, PRED_UE_LOW, PRED_UE_HIGH)
    else:
        pred_df['ue_fin'] = pred_df['ue'].copy()

if ref_has_ue:
    if USE_UASTHN2:
        ref_df['ue_fin'] = build_ue_fin(ref_df, REF_UE_LOW, REF_UE_HIGH)
    else:
        ref_df['ue_fin'] = ref_df['ue'].copy()


# ==============================================================================
# sss ITERATION PATH COLUMN DETECTION
# ==============================================================================
ITER_SUFFIXES     = ['_1','_2','_3','_4','_5','_6']
iter_cols_present = [
    s for s in ITER_SUFFIXES
    if all(f"{c}{s}" in pred_df.columns for c in ['x1','y1','x2','y2','x3','y3','x4','y4'])
]
HAS_ITER_PATH = len(iter_cols_present) > 0


# ==============================================================================
# sss DISTANCE / DISPLACEMENT (iter0 -> iter6) COMPUTATION
# ==============================================================================
# These are used in the dist/disp vs UE heatmap section. We need the image size
# per row to compute centers relative to image-center (iteration 0).
pred_dist_vals = np.full(len(pred_df), np.nan)
pred_disp_vals = np.full(len(pred_df), np.nan)

if HAS_ITER_PATH:
    for idx, row in pred_df.iterrows():
        sat_path = resolve_img_path(row["sat"])
        if sat_path is None:
            continue
        try:
            with Image.open(sat_path) as im:
                img_w, img_h = im.size
        except Exception:
            continue
        d, disp = compute_iter_dist_disp(row, img_h, img_w)
        pred_dist_vals[idx] = d
        pred_disp_vals[idx] = disp

pred_df['_iter_dist'] = pred_dist_vals
pred_df['_iter_disp'] = pred_disp_vals


# ==============================================================================
# sss DISTANCE DATAFRAME (CE / MACE per row, vs Ground Truth)
# ==============================================================================
distance_data = []
for idx, row_pred in pred_df.iterrows():
    row_ref = ref_df.iloc[idx] if ref_df is not None and idx < len(ref_df) else None
    row_gt  = gt_df.iloc[idx]  if gt_df  is not None and idx < len(gt_df)  else None

    row_data = {'img_idx': idx}
    if row_gt is not None:
        d = compute_distances(row_gt, row_pred)
        row_data.update({
            'pred_ce': d['center_dist'], 'pred_mace': d['avg_corner_dist'],
            'pred_max_corner': d['max_corner_dist'], 'pred_min_corner': d['min_corner_dist'],
        })
    if row_gt is not None and row_ref is not None:
        d = compute_distances(row_gt, row_ref)
        row_data.update({
            'ref_ce': d['center_dist'], 'ref_mace': d['avg_corner_dist'],
            'ref_max_corner': d['max_corner_dist'], 'ref_min_corner': d['min_corner_dist'],
        })
    distance_data.append(row_data)

dist_df = pd.DataFrame(distance_data)


# ==============================================================================
# sss DISTANCE REPORT — STHN baseline + UASTHN-1 / UASTHN-2 acc/rej/sus breakdown
# ==============================================================================
print("\n" + "="*130)
print("DISTANCE REPORT (in pixels)")
print("  STHN row = raw model performance (no UE-based filtering)")
if USE_UASTHN2:
    print(f"  {NAME_1T} rows = performance split by UASTHN-1 decision (UE < thresh -> Accepted)")
    print(f"  {NAME_2T} rows = performance split by UASTHN-2 decision (3-zone + ue_fin)")
    print(f"  Suspicious rows = same data, shown twice: once classified by UE, once by UE_fin")
else:
    print(f"  {NAME_1T} rows = performance split by UASTHN-1 decision (UE < thresh -> Accepted)")
    print(f"  ({NAME_2T} / suspicious rows skipped because USE_UASTHN2 = False)")
print("CE = Center Error | MACE = Mean Absolute Corner Error")
print(f"PRED = {PRED_NAME}")
print(f"REF  = {REF_NAME}")
print("-"*130)

dist_report_headers = ["Method", "Source", "Subset", "Count",
                        "MACE Avg", "MACE Std", "CE Avg", "CE Std", "CE Max",
                        f"Accurate (CE≤{CE_THRESH})", f"Inaccurate (CE>{CE_THRESH})"]

dist_report_rows = []

def ce_mace_row(method, source, subset, ce_vals, mace_vals):
    n = len(ce_vals)
    if n == 0:
        return [method, source, subset, "0", "—", "—", "—", "—", "—", "—", "—"]
    acc = int(np.sum(ce_vals <= CE_THRESH))
    return [method, source, subset, str(n),
            f"{mace_vals.mean():.2f}", f"{mace_vals.std():.2f}",
            f"{ce_vals.mean():.2f}", f"{ce_vals.std():.2f}", f"{ce_vals.max():.2f}",
            f"{acc} ({100*acc/n:.1f}%)", f"{n-acc} ({100*(n-acc)/n:.1f}%)"]


def build_distance_rows(src_label, ce_col, mace_col, ue_df,
                         ue1_thresh, ue_low, ue_high, ue2_mid, has_ue_):
    rows = []
    ce_all   = dist_df[ce_col].values
    mace_all = dist_df[mace_col].values
    valid    = ~np.isnan(ce_all)
    ce_all, mace_all = ce_all[valid], mace_all[valid]

    # STHN baseline (all data, no filtering)
    rows.append(ce_mace_row(NAME_STHN, src_label, "All", ce_all, mace_all))

    if not has_ue_:
        return rows

    ue_vals = ue_df['ue'].values[:len(dist_df)][valid]

    # ── UASTHN-1: accepted / rejected by raw ue vs ue1_thresh ──
    acc1_mask = ue_vals < ue1_thresh
    rows.append(ce_mace_row(f"{NAME_1T} (UE<{ue1_thresh})", src_label, "Acc",
                             ce_all[acc1_mask], mace_all[acc1_mask]))
    rows.append(ce_mace_row(f"{NAME_1T} (UE>={ue1_thresh})", src_label, "Rej",
                             ce_all[~acc1_mask], mace_all[~acc1_mask]))

    if not USE_UASTHN2:
        return rows

    ue_fin_vals = ue_df['ue_fin'].values[:len(dist_df)][valid]
    sus_mask    = (ue_vals >= ue_low) & (ue_vals <= ue_high)

    # ── UASTHN-2 final accept/reject (non-sus via ue, sus via ue_fin) ──
    final_acc = np.zeros(len(ce_all), dtype=bool)
    final_acc[~sus_mask] = ue_vals[~sus_mask] < ue_low
    final_acc[sus_mask]  = ue_fin_vals[sus_mask] < ue2_mid

    rows.append(ce_mace_row(f"{NAME_2T} (UE_fin<{ue2_mid})", src_label, "Acc",
                             ce_all[final_acc], mace_all[final_acc]))
    rows.append(ce_mace_row(f"{NAME_2T} (UE_fin>={ue2_mid})", src_label, "Rej",
                             ce_all[~final_acc], mace_all[~final_acc]))

    # ── Suspicious subset: same rows, classified twice ──
    sus_ce, sus_mace   = ce_all[sus_mask], mace_all[sus_mask]
    sus_ue, sus_ue_fin = ue_vals[sus_mask], ue_fin_vals[sus_mask]

    sus_acc_by_ue     = sus_ue     < ue2_mid     # by construction this is mostly False
    sus_acc_by_uefin  = sus_ue_fin < ue2_mid

    rows.append(ce_mace_row(f"{NAME_2T} ({ue_low}<=UE<{ue_high})", src_label,
                             f"Sus All",
                             sus_ce, sus_mace))
    rows.append(ce_mace_row(f"  -> Before (UE<{ue2_mid})", src_label, "Sus Acc",
                             sus_ce[sus_acc_by_ue], sus_mace[sus_acc_by_ue]))
    rows.append(ce_mace_row(f"  -> Before (UE>={ue2_mid})", src_label, "Sus Rej",
                             sus_ce[~sus_acc_by_ue], sus_mace[~sus_acc_by_ue]))

    
    rows.append(ce_mace_row(f"  -> After (UE_fin<{ue2_mid})", src_label, "Sus Acc",
                             sus_ce[sus_acc_by_uefin], sus_mace[sus_acc_by_uefin]))
    rows.append(ce_mace_row(f"  -> After (UE_fin>={ue2_mid})", src_label, "Sus Rej",
                             sus_ce[~sus_acc_by_uefin], sus_mace[~sus_acc_by_uefin]))

    return rows


if 'pred_ce' in dist_df.columns:
    dist_report_rows += build_distance_rows(
        f"PRED", 'pred_ce', 'pred_mace', pred_df,
        PRED_UE1_THRESH, PRED_UE_LOW, PRED_UE_HIGH, PRED_UE2_MID, pred_has_ue)

if USE_REFERENCE and ref_df is not None and 'ref_ce' in dist_df.columns:
    dist_report_rows += build_distance_rows(
        f"REF", 'ref_ce', 'ref_mace', ref_df,
        REF_UE1_THRESH, REF_UE_LOW, REF_UE_HIGH, REF_UE2_MID, ref_has_ue)

if dist_report_rows:
    cw = [max(len(h), max(len(r[i]) for r in dist_report_rows))
          for i, h in enumerate(dist_report_headers)]
    hl = " | ".join(h.ljust(cw[i]) for i, h in enumerate(dist_report_headers))
    print(hl)
    print("-" * len(hl))
    prev_src = None
    for row in dist_report_rows:
        if prev_src and prev_src != row[1]:
            print("-" * len(hl))
        print(" | ".join(cell.ljust(cw[i]) for i, cell in enumerate(row)))
        prev_src = row[1]
    print("="*130)
    print(f"Total images analyzed: {len(dist_df)}")
    print("="*130 + "\n")

# ==============================================================================
# sss CE DISTRIBUTION PLOT (general, all data — vertical lines only)
# ==============================================================================
ce_plot_configs = []
if 'pred_ce' in dist_df.columns:
    ce_plot_configs.append((dist_df['pred_ce'].dropna().values,
                            f"PRED — {NAME_STHN}\nCenter Error Distribution"))
if ref_df is not None and 'ref_ce' in dist_df.columns:
    ce_plot_configs.append((dist_df['ref_ce'].dropna().values,
                            f"REF — {NAME_STHN}\nCenter Error Distribution"))

if ce_plot_configs:
    fig_ce, axes_ce = plt.subplots(1, len(ce_plot_configs), figsize=(7 * len(ce_plot_configs), 5))
    if len(ce_plot_configs) == 1:
        axes_ce = [axes_ce]
    colors_ce = ['#9c27b0', '#009688']
    for ax, (ce_vals, title), col in zip(axes_ce, ce_plot_configs, colors_ce):
        n      = len(ce_vals)
        n_bins = min(40, max(10, n // 3))
        counts, bin_edges, patches = ax.hist(ce_vals, bins=n_bins, color=col,
                                              alpha=0.75, edgecolor='white', linewidth=0.5)
        mean_val   = ce_vals.mean()
        median_val = np.median(ce_vals)
        xspan = bin_edges[-1] - bin_edges[0]
        ax.axvline(mean_val,   color='navy',      linestyle='--', linewidth=2,   label=f'Mean: {mean_val:.2f}')
        ax.axvline(median_val, color='darkorange', linestyle=':',  linewidth=2.2, label=f'Median: {median_val:.2f}')
        ax.axvline(CE_THRESH,  color='black',      linestyle='-',  linewidth=1.8, label=f'CE Threshold: {CE_THRESH}')
        ymax = counts.max()
        ax.annotate(f'Mean\n{mean_val:.1f}',
                    xy=(mean_val, ymax * 0.92),
                    xytext=(mean_val + xspan * 0.04, ymax * 0.92),
                    color='navy', fontsize=9, arrowprops=dict(arrowstyle='->', color='navy', lw=1.2))
        ax.annotate(f'Median\n{median_val:.1f}',
                    xy=(median_val, ymax * 0.75),
                    xytext=(median_val + xspan * 0.04, ymax * 0.75),
                    color='darkorange', fontsize=9, arrowprops=dict(arrowstyle='->', color='darkorange', lw=1.2))
        ax.set_yscale('log')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel("Center Error (pixels)", fontsize=11)
        ax.set_ylabel("Count (log scale)", fontsize=11)
        ax.legend(fontsize=8.5, loc='upper right')
        ax.grid(axis='y', alpha=0.3, which='both')
        n_acc = np.sum(ce_vals <= CE_THRESH)
        ax.text(0.02, 0.97,
                f"✓ Accurate  : {n_acc} ({100*n_acc/n:.1f}%)\n"
                f"✗ Inaccurate: {n-n_acc} ({100*(n-n_acc)/n:.1f}%)",
                transform=ax.transAxes, fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.85))
    fig_ce.suptitle(f"Spatial Center Error (CE) Distribution — {NAME_STHN} (all data)  |  CE Threshold = {CE_THRESH} px\n"
                    f"PRED ({PRED_NAME}), REF ({REF_NAME})",
                    fontsize=14, fontweight='bold', y=1)
    fig_ce.tight_layout(h_pad=66)
    plt.show()


# ==============================================================================
# sss CE DISTRIBUTION BY ACCEPT/REJECT — twin-axis plots
#   Rows: UASTHN-1 | UASTHN-2 (final) | Suspicious-by-UE | Suspicious-by-UE_fin
#   Cols: PRED | REF
#   Each cell: histogram of CE values, twin y-axes
#     left  axis (green) = Accepted CE distribution
#     right axis (red)   = Rejected CE distribution
# ==============================================================================
def plot_twin_ce(ax_left, ce_acc, ce_rej, title, bin_count=25, log=True):
    """Twin-axis histogram: left=accepted(green), right=rejected(red)."""
    ax_right = ax_left.twinx()

    all_vals = np.concatenate([ce_acc, ce_rej]) if (len(ce_acc) + len(ce_rej)) > 0 else np.array([0, 1])
    bins = np.linspace(all_vals.min(), max(all_vals.max(), all_vals.min() + 1), bin_count)
    bin_width = bins[1] - bins[0]

    total = len(ce_acc) + len(ce_rej)
    if len(ce_acc) > 0:
        ax_left.hist(ce_acc, bins=bins, color='#4caf50', alpha=0.6, edgecolor='white',
                      label=f'Accepted {len(ce_acc)} ({100*len(ce_acc)/total:.2f}%)')
    if len(ce_rej) > 0:
        ax_right.hist(ce_rej, bins=bins, color='#f44336', alpha=0.5, edgecolor='white',
                       label=f'Rejected {len(ce_rej)} ({100*len(ce_rej)/total:.2f}%)')

    ax_left.axvline(CE_THRESH, color='black', linestyle='--', linewidth=1.5,
                     label=f'CE Thresh={CE_THRESH}')

    ax_left.set_ylabel(f"Count{' (log)' if log else ''}: Accepted", color='#2e7d32', fontsize=9)
    ax_right.set_ylabel(f"Count{' (log)' if log else ''}: Rejected", color='#c62828', fontsize=9)
    if log:
        ax_left.set_yscale('log')
        ax_right.set_yscale('log')
    ax_left.tick_params(axis='y', labelcolor='#2e7d32')
    ax_right.tick_params(axis='y', labelcolor='#c62828')
    ax_left.set_xlabel(f"Center Error (px)  |  bar width ≈ {bin_width:.1f}px", fontsize=8.5)
    ax_left.set_title(title, fontsize=10, fontweight='bold')

    handles1, labels1 = ax_left.get_legend_handles_labels()
    handles2, labels2 = ax_right.get_legend_handles_labels()
    ax_left.legend(handles1 + handles2, labels1 + labels2, fontsize=7.5, loc='upper right')
    ax_left.grid(axis='y', alpha=0.3, color='#2e7d32')
    ax_right.grid(axis='y', alpha=0.3, color='#c62828')


if USE_UASTHN2:
    ce_rows_n = 4
    row_titles = [
        f"{NAME_1T}: Accepted vs Rejected CE",
        f"{NAME_2T}: Accepted vs Rejected CE (final)",
        f"Suspicious data — Acc/Rej by UE (low={{LOW}})",
        f"Suspicious data — same rows, Acc/Rej by UE_fin (mid={{MID}})",
    ]
else:
    ce_rows_n = 1
    row_titles = [f"{NAME_1T}: Accepted vs Rejected CE"]

n_ce_cols = 1 + (1 if USE_REFERENCE and ref_df is not None and 'ref_ce' in dist_df.columns else 0)

fig_ce2, axes_ce2 = plt.subplots(ce_rows_n, n_ce_cols, figsize=(7 * n_ce_cols, 4.2 * ce_rows_n), squeeze=False)

def get_ce_acc_rej_masks(src_df, ce_col, ue1_thresh, ue_low, ue_high, ue2_mid, has_ue_):
    """
    Returns dict with masks for each of the 4 rows, aligned to dist_df rows (NaN-CE dropped).
    """
    ce_all = dist_df[ce_col].values
    valid  = ~np.isnan(ce_all)
    ce_v   = ce_all[valid]

    out = {'ce': ce_v}
    if not has_ue_:
        return out

    ue_v = src_df['ue'].values[:len(dist_df)][valid]
    out['uasthn1_acc'] = ue_v < ue1_thresh

    if not USE_UASTHN2:
        return out

    ue_fin_v = src_df['ue_fin'].values[:len(dist_df)][valid]
    sus_mask = (ue_v >= ue_low) & (ue_v <= ue_high)

    final_acc = np.zeros(len(ce_v), dtype=bool)
    final_acc[~sus_mask] = ue_v[~sus_mask] < ue_low
    final_acc[sus_mask]  = ue_fin_v[sus_mask] < ue2_mid
    out['uasthn2_acc'] = final_acc

    out['sus_mask']        = sus_mask
    out['sus_acc_by_ue']    = ue_v[sus_mask] < ue2_mid
    out['sus_acc_by_uefin'] = ue_fin_v[sus_mask] < ue2_mid
    return out


col_specs = []
if 'pred_ce' in dist_df.columns:
    col_specs.append((f"PRED", pred_df, 'pred_ce',
                       PRED_UE1_THRESH, PRED_UE_LOW, PRED_UE_HIGH, PRED_UE2_MID, pred_has_ue))
if n_ce_cols == 2 and ref_df is not None and 'ref_ce' in dist_df.columns:
    col_specs.append((f"REF", ref_df, 'ref_ce',
                       REF_UE1_THRESH, REF_UE_LOW, REF_UE_HIGH, REF_UE2_MID, ref_has_ue))

for col_i, (src_label, src_df, ce_col, ue1_t, ue_low, ue_high, ue2_mid, has_ue_) in enumerate(col_specs):
    masks = get_ce_acc_rej_masks(src_df, ce_col, ue1_t, ue_low, ue_high, ue2_mid, has_ue_)
    ce_v  = masks['ce']

    # Row 0: UASTHN-1
    if 'uasthn1_acc' in masks:
        m = masks['uasthn1_acc']
        plot_twin_ce(axes_ce2[0][col_i], ce_v[m], ce_v[~m],
                     f"{src_label} — {row_titles[0]} (thr={ue1_t})")
    else:
        axes_ce2[0][col_i].axis('off')

    if not USE_UASTHN2:
        continue

    # Row 1: UASTHN-2 final
    if 'uasthn2_acc' in masks:
        m = masks['uasthn2_acc']
        plot_twin_ce(axes_ce2[1][col_i], ce_v[m], ce_v[~m],
                     f"{src_label} — {row_titles[1]} (low={ue_low},high={ue_high},mid={ue2_mid})")
    else:
        axes_ce2[1][col_i].axis('off')

    # Row 2: suspicious by UE
    if 'sus_mask' in masks:
        sus_ce = ce_v[masks['sus_mask']]
        m      = masks['sus_acc_by_ue']
        plot_twin_ce(axes_ce2[2][col_i], sus_ce[m], sus_ce[~m],
                     f"{src_label} — {row_titles[2].format(LOW=ue_low)}")
        # Row 3: same suspicious rows, by UE_fin
        m2 = masks['sus_acc_by_uefin']
        plot_twin_ce(axes_ce2[3][col_i], sus_ce[m2], sus_ce[~m2],
                     f"{src_label} — {row_titles[3].format(MID=ue2_mid)}")
    else:
        axes_ce2[2][col_i].axis('off')
        axes_ce2[3][col_i].axis('off')

# Turn off any unused axes (e.g. if only PRED column exists)
for r in range(ce_rows_n):
    for c in range(n_ce_cols):
        if c >= len(col_specs):
            axes_ce2[r][c].axis('off')

fig_ce2.suptitle(
    f"CE Distribution by Accept/Reject Decision  |  CE Threshold = {CE_THRESH} px\n"
    f"Twin axes: left(green)=Accepted, right(red)=Rejected\n"
    f"PRED ({PRED_NAME}), REF ({REF_NAME})",
    fontsize=12, fontweight='bold', y=1)
fig_ce2.tight_layout()
plt.show()

# ==============================================================================
# sss UNCERTAINTY ESTIMATION (UE) REPORT — statistics table
#   For ALL / Trues (CE<=thresh) / Falses (CE>thresh), report UE stats for
#   data classified as Accepted / Rejected / Suspicious by UASTHN-1 and UASTHN-2.
# ==============================================================================
ue_stats_headers = [
    "Source", "Method", "Data", "Count",
    "Accept", "Reject",
    "Min", "Mean", "Median", "Max", "Std"
]

def fmt_stat(v):
    return "—" if (v is None or (isinstance(v, float) and np.isnan(v))) else f"{v:.2f}"


def stat_row(source, method, data_label,
             accept_mask, reject_mask, vals):

    s = series_stats(vals)
    n_total = len(vals)

    n_acc = int(np.sum(accept_mask))
    n_rej = int(np.sum(reject_mask))

    acc_txt = f"{n_acc} ({100*n_acc/n_total:.1f}%)" if n_total > 0 else "0 (0.0%)"
    rej_txt = f"{n_rej} ({100*n_rej/n_total:.1f}%)" if n_total > 0 else "0 (0.0%)"

    return [
        source,
        method,
        data_label,
        str(s['n']),
        acc_txt,
        rej_txt,
        fmt_stat(s['min']),
        fmt_stat(s['mean']),
        fmt_stat(s['median']),
        fmt_stat(s['max']),
        fmt_stat(s['std'])
    ]


def build_ue_stats_rows(src_label, src_df, ce_col,
                        ue1_thresh, ue_low, ue_high,
                        ue2_mid, has_ue_):

    rows = []

    if not has_ue_:
        return rows

    ce_all = dist_df[ce_col].values
    valid = ~np.isnan(ce_all)

    ce_v = ce_all[valid]
    ue_v = src_df['ue'].values[:len(dist_df)][valid]

    true_mask = ce_v <= CE_THRESH
    false_mask = ~true_mask

    # ==========================================================
    # UASTHN-1
    # ==========================================================
    acc1 = ue_v < ue1_thresh
    rej1 = ~acc1

    for data_label in ["All", "Trues", "Falses"]:

        if data_label == "All":
            mask_t = np.ones_like(acc1, dtype=bool)
        elif data_label == "Trues":
            mask_t = true_mask
        else:
            mask_t = false_mask

        sel = mask_t

        rows.append(
            stat_row(
                src_label,
                f"{NAME_1T} (Acc<{ue1_thresh}<=Rej)",
                data_label,
                acc1[sel],
                rej1[sel],
                ue_v[sel]
            )
        )

    if not USE_UASTHN2:
        return rows

    # ==========================================================
    # UASTHN-2
    # ==========================================================
    ue_fin_v = src_df['ue_fin'].values[:len(dist_df)][valid]

    sus_mask = (ue_v >= ue_low) & (ue_v <= ue_high)

    final_acc = np.zeros(len(ue_v), dtype=bool)
    final_acc[~sus_mask] = ue_v[~sus_mask] < ue_low
    final_acc[sus_mask] = ue_fin_v[sus_mask] < ue2_mid

    final_rej = ~final_acc

    tag2 = f"{NAME_2T} (Acc<{ue2_mid}<=Rej)"

    for data_label in ["All", "Trues", "Falses"]:

        if data_label == "All":
            mask_t = np.ones_like(final_acc, dtype=bool)
        elif data_label == "Trues":
            mask_t = true_mask
        else:
            mask_t = false_mask

        sel = mask_t

        rows.append(
            stat_row(
                src_label,
                tag2,
                data_label,
                final_acc[sel],
                final_rej[sel],
                ue_fin_v[sel]
            )
        )

    # ==========================================================
    # Suspicious subset — based on UE (raw)
    # ==========================================================
    sus_ue     = ue_v[sus_mask]
    sus_ue_fin = ue_fin_v[sus_mask]

    sus_acc_ue = sus_ue_fin < ue2_mid   # decision is always made via ue_fin
    sus_rej_ue = ~sus_acc_ue

    sus_true  = true_mask[sus_mask]
    sus_false = false_mask[sus_mask]

    for data_label, mask_t in [
        ("Sus All", np.ones_like(sus_acc_ue, dtype=bool)),
        ("Sus Trues", sus_true),
        ("Sus Falses", sus_false)
    ]:

        rows.append(
            stat_row(
                src_label,
                f"  -> Before (Acc<{ue2_mid}<=Rej)",
                data_label,
                sus_acc_ue[mask_t],
                sus_rej_ue[mask_t],
                sus_ue[mask_t]
            )
        )

    # ==========================================================
    # Suspicious subset — same rows, based on UE_fin
    # ==========================================================
    sus_acc_uefin = sus_ue_fin < ue2_mid
    sus_rej_uefin = ~sus_acc_uefin

    for data_label, mask_t in [
        ("Sus All", np.ones_like(sus_acc_uefin, dtype=bool)),
        ("Sus Trues", sus_true),
        ("Sus Falses", sus_false)
    ]:

        rows.append(
            stat_row(
                src_label,
                f"  -> After (Acc<{ue2_mid}<=Rej)",
                data_label,
                sus_acc_uefin[mask_t],
                sus_rej_uefin[mask_t],
                sus_ue_fin[mask_t]
            )
        )

    return rows


# ==============================================================================
# Build rows
# ==============================================================================
ue_stats_rows = []

if pred_has_ue:
    ue_stats_rows += build_ue_stats_rows(
        "PRED", pred_df, 'pred_ce',
        PRED_UE1_THRESH, PRED_UE_LOW,
        PRED_UE_HIGH, PRED_UE2_MID,
        pred_has_ue
    )

if USE_REFERENCE and ref_has_ue:
    ue_stats_rows += build_ue_stats_rows(
        "REF", ref_df, 'ref_ce',
        REF_UE1_THRESH, REF_UE_LOW,
        REF_UE_HIGH, REF_UE2_MID,
        ref_has_ue
    )

# ==============================================================================
# Print
# ==============================================================================
if ue_stats_rows:

    print("\n" + "="*150)
    print("UNCERTAINTY ESTIMATION (UE) REPORT — statistics")

    print(
        f"  {NAME_1T}: "
        f"PRED UE<{PRED_UE1_THRESH}→Accepted | "
        f"REF UE<{REF_UE1_THRESH}→Accepted"
    )

    if USE_UASTHN2:
        print(
            f"  {NAME_2T}: "
            f"PRED low={PRED_UE_LOW} high={PRED_UE_HIGH} mid={PRED_UE2_MID}"
            f" | REF low={REF_UE_LOW} high={REF_UE_HIGH} mid={REF_UE2_MID}"
        )
        print(f"  'Suspicious' rows show the SAME data twice: once classified by raw UE, "
              f"once by UE_fin (decisions in both cases use UE_fin < mid)")

    print("  Trues = CE<=thresh | Falses = CE>thresh")
    print("PRED =", PRED_NAME)
    print("REF  =", REF_NAME)
    print("="*150)

    cw = [
        max(len(h), max(len(str(r[i])) for r in ue_stats_rows))
        for i, h in enumerate(ue_stats_headers)
    ]

    hl = " | ".join(h.ljust(cw[i]) for i, h in enumerate(ue_stats_headers))
    print(hl)
    print("-" * len(hl))

    prev_source = None

    for row in ue_stats_rows:
        source = row[0]

        if prev_source is not None and prev_source != source:
            print("-" * len(hl))

        print(
            " | ".join(
                str(cell).ljust(cw[i])
                for i, cell in enumerate(row)
            )
        )

        prev_source = source

    print("="*150 + "\n")

# ==============================================================================
# sss UE DISTRIBUTION — general (all data, vertical lines only)
#   One combined row per source: shows ue (and ue_fin if UASTHN2) distributions
#   with threshold lines, no area shading.
# ==============================================================================
def plot_ue_general(ax, ue_vals, ue_fin_vals, title, ue1_thresh, ue_low, ue_high, ue2_mid):
    n = len(ue_vals)
    n_bins = min(60, max(10, n // 3))
    ax.hist(ue_vals, bins=n_bins, color='steelblue', alpha=0.6,
            edgecolor='white', linewidth=0.4, label=f'UE (n={n})')

    if USE_UASTHN2 and ue_fin_vals is not None and len(ue_fin_vals) > 0:
        ax.hist(ue_fin_vals, bins=n_bins, color='mediumpurple', alpha=0.45,
                edgecolor='white', linewidth=0.4, label=f'UE_fin (n={len(ue_fin_vals)})')

    ax.axvline(ue1_thresh, color='black', linestyle='--', linewidth=1.8,
               label=f'{NAME_1T} thresh = {ue1_thresh}')
    if USE_UASTHN2:
        ax.axvline(ue_low,  color='green', linestyle=':',  linewidth=1.6, label=f'UE_LOW = {ue_low}')
        ax.axvline(ue_high, color='red',   linestyle=':',  linewidth=1.6, label=f'UE_HIGH = {ue_high}')
        ax.axvline(ue2_mid, color='purple', linestyle='-.', linewidth=1.6, label=f'UE2_MID = {ue2_mid}')

    mv, mdv = ue_vals.mean(), np.median(ue_vals)
    ax.axvline(mv,  color='navy',       linestyle='-', linewidth=1.3, alpha=0.7, label=f'Mean: {mv:.2f}')
    ax.axvline(mdv, color='darkorange', linestyle='-', linewidth=1.3, alpha=0.7, label=f'Median: {mdv:.2f}')

    ax.set_yscale('log')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel("UE Value", fontsize=10)
    ax.set_ylabel("Count (log scale)", fontsize=10)
    ax.legend(fontsize=7, loc='upper right', ncol=2)
    ax.grid(axis='y', alpha=0.3, which='both')


n_ue_cols = 1 + (1 if USE_REFERENCE and ref_has_ue else 0)

if pred_has_ue or ref_has_ue:
    fig_ueg, axes_ueg = plt.subplots(1, n_ue_cols, figsize=(7 * n_ue_cols, 5), squeeze=False)

    if pred_has_ue:
        plot_ue_general(axes_ueg[0][0], pred_df['ue'].dropna().values,
                        pred_df['ue_fin'].dropna().values if USE_UASTHN2 else None,
                        f"PRED — UE Distribution (general)",
                        PRED_UE1_THRESH, PRED_UE_LOW, PRED_UE_HIGH, PRED_UE2_MID)
    else:
        axes_ueg[0][0].axis('off')

    if n_ue_cols == 2:
        if ref_has_ue:
            plot_ue_general(axes_ueg[0][1], ref_df['ue'].dropna().values,
                            ref_df['ue_fin'].dropna().values if USE_UASTHN2 else None,
                            f"REF — UE Distribution (general)",
                            REF_UE1_THRESH, REF_UE_LOW, REF_UE_HIGH, REF_UE2_MID)
        else:
            axes_ueg[0][1].axis('off')

    fig_ueg.suptitle("UE Distribution — general (all data)\n"
                     f"PRED ({PRED_NAME}), REF ({REF_NAME})", fontsize=13, fontweight='bold')
    fig_ueg.tight_layout()
    plt.show()


# ==============================================================================
# sss UE DISTRIBUTION BY TRUE/FALSE — twin-axis plots
#   Rows: UASTHN-1 | UASTHN-2 (ue_fin) | Suspicious-by-UE | Suspicious-by-UE_fin
#   Cols: PRED | REF
#   Each cell: left axis(green)=Trues (CE<=thresh), right axis(red)=Falses (CE>thresh)
# ==============================================================================
def plot_twin_ue(ax_left, ue_true, ue_false, title, bin_count=25, log=True):
    ax_right = ax_left.twinx()

    all_vals = np.concatenate([ue_true, ue_false]) if (len(ue_true) + len(ue_false)) > 0 else np.array([0, 1])
    lo, hi = all_vals.min(), max(all_vals.max(), all_vals.min() + 1)
    bins = np.linspace(lo, hi, bin_count)
    bin_width = bins[1] - bins[0]

    if len(ue_true) > 0:
        ax_left.hist(ue_true, bins=bins, color='#4caf50', alpha=0.6, edgecolor='white',
                      label=f'Trues, CE≤{CE_THRESH} (n={len(ue_true)})')
    if len(ue_false) > 0:
        ax_right.hist(ue_false, bins=bins, color='#f44336', alpha=0.5, edgecolor='white',
                       label=f'Falses, CE>{CE_THRESH} (n={len(ue_false)})')

    ax_left.set_ylabel(f"Count{' (log)' if log else ''}: Trues", color='#2e7d32', fontsize=9)
    ax_right.set_ylabel(f"Count{' (log)' if log else ''}: Falses", color='#c62828', fontsize=9)
    if log:
        ax_left.set_yscale('log')
        ax_right.set_yscale('log')
    ax_left.tick_params(axis='y', labelcolor='#2e7d32')
    ax_right.tick_params(axis='y', labelcolor='#c62828')
    ax_left.set_xlabel(f"UE Value  |  bar width ≈ {bin_width:.2f}", fontsize=8.5)
    ax_left.set_title(title, fontsize=10, fontweight='bold')

    handles1, labels1 = ax_left.get_legend_handles_labels()
    handles2, labels2 = ax_right.get_legend_handles_labels()
    ax_left.legend(handles1 + handles2, labels1 + labels2, fontsize=7.5, loc='upper right')
    ax_left.grid(axis='y', alpha=0.3, color='#2e7d32')
    ax_right.grid(axis='y', alpha=0.3, color='#c62828')


if USE_UASTHN2:
    ue_rows_n = 4
    ue_row_titles = [
        f"{NAME_1T}: UE by Trues/Falses",
        f"{NAME_2T}: UE_fin by Trues/Falses",
        f"Suspicious — UE by Trues/Falses",
        f"Suspicious (same rows) — UE_fin by Trues/Falses",
    ]
else:
    ue_rows_n = 1
    ue_row_titles = [f"{NAME_1T}: UE by Trues/Falses"]

n_ue2_cols = 1 + (1 if USE_REFERENCE and ref_has_ue else 0)
fig_uetf, axes_uetf = plt.subplots(ue_rows_n, n_ue2_cols, figsize=(7 * n_ue2_cols, 4.2 * ue_rows_n), squeeze=False)


def get_ue_truefalse_data(src_df, ce_col, ue1_thresh, ue_low, ue_high, ue2_mid, has_ue_):
    ce_all = dist_df[ce_col].values
    valid  = ~np.isnan(ce_all)
    ce_v   = ce_all[valid]
    out = {'ce': ce_v}
    if not has_ue_:
        return out
    ue_v = src_df['ue'].values[:len(dist_df)][valid]
    out['ue'] = ue_v
    out['true_mask'] = ce_v <= CE_THRESH

    if not USE_UASTHN2:
        return out

    ue_fin_v = src_df['ue_fin'].values[:len(dist_df)][valid]
    out['ue_fin'] = ue_fin_v
    sus_mask = (ue_v >= ue_low) & (ue_v <= ue_high)
    out['sus_mask'] = sus_mask
    return out


col_specs_ue = []
if pred_has_ue:
    col_specs_ue.append((f"PRED", pred_df, 'pred_ce',
                          PRED_UE1_THRESH, PRED_UE_LOW, PRED_UE_HIGH, PRED_UE2_MID, pred_has_ue))
if n_ue2_cols == 2 and ref_has_ue:
    col_specs_ue.append((f"REF", ref_df, 'ref_ce',
                          REF_UE1_THRESH, REF_UE_LOW, REF_UE_HIGH, REF_UE2_MID, ref_has_ue))

for col_i, (src_label, src_df, ce_col, ue1_t, ue_low, ue_high, ue2_mid, has_ue_) in enumerate(col_specs_ue):
    d = get_ue_truefalse_data(src_df, ce_col, ue1_t, ue_low, ue_high, ue2_mid, has_ue_)
    if 'ue' not in d:
        for r in range(ue_rows_n):
            axes_uetf[r][col_i].axis('off')
        continue

    tmask = d['true_mask']

    # Row 0: UASTHN-1 -- ue by true/false
    plot_twin_ue(axes_uetf[0][col_i], d['ue'][tmask], d['ue'][~tmask],
                 f"{src_label} — {ue_row_titles[0]}")

    if not USE_UASTHN2:
        continue

    # Row 1: UASTHN-2 -- ue_fin by true/false (all data)
    plot_twin_ue(axes_uetf[1][col_i], d['ue_fin'][tmask], d['ue_fin'][~tmask],
                 f"{src_label} — {ue_row_titles[1]}")

    # Row 2: suspicious -- ue by true/false (same suspicious rows)
    sus = d['sus_mask']
    sus_ue, sus_uefin, sus_t = d['ue'][sus], d['ue_fin'][sus], tmask[sus]
    plot_twin_ue(axes_uetf[2][col_i], sus_ue[sus_t], sus_ue[~sus_t],
                 f"{src_label} — {ue_row_titles[2]} (range {ue_low}-{ue_high})")

    # Row 3: same suspicious rows -- ue_fin by true/false
    plot_twin_ue(axes_uetf[3][col_i], sus_uefin[sus_t], sus_uefin[~sus_t],
                 f"{src_label} — {ue_row_titles[3]}")

for r in range(ue_rows_n):
    for c in range(n_ue2_cols):
        if c >= len(col_specs_ue):
            axes_uetf[r][c].axis('off')

fig_uetf.suptitle(
    f"UE Distribution by Spatial Correctness  |  CE Threshold = {CE_THRESH} px\n"
    f"Twin axes: left(green)=Trues, right(red)=Falses\n"
    f"PRED ({PRED_NAME}), REF ({REF_NAME})",
    fontsize=12, fontweight='bold', y=1)
fig_uetf.tight_layout()
plt.show()

# ==============================================================================
# sss UNCERTAINTY EVALUATION VS SPATIAL ERROR (CE) — aggregated table
#   True Positive  = model accepts AND CE<=thresh
#   True Negative  = model rejects AND CE>thresh
# ==============================================================================
cm_results = {}

def build_eval_df(source_df, ce_col):
    if 'ue' not in source_df.columns or ce_col not in dist_df.columns:
        return None
    d = pd.DataFrame({
        'ue':     source_df['ue'].values[:len(dist_df)],
        'ue_fin': source_df['ue_fin'].values[:len(dist_df)],
        'ce':     dist_df[ce_col].values,
    })
    return d.dropna(subset=['ue', 'ce'])

eval_pred = build_eval_df(pred_df, 'pred_ce') if pred_has_ue else None
eval_ref  = build_eval_df(ref_df,  'ref_ce')  if (USE_REFERENCE and ref_has_ue) else None

if eval_pred is not None:
    ce_real = eval_pred['ce'] <= CE_THRESH
    cm_results[('pred', '1t')] = cm_metrics(eval_pred['ue'] < PRED_UE1_THRESH, ce_real)
    if USE_UASTHN2:
        cm_results[('pred', '2t')] = cm_metrics(
            compute_final_accept_2t(eval_pred, PRED_UE_LOW, PRED_UE_HIGH, PRED_UE2_MID), ce_real)

if eval_ref is not None:
    ce_real = eval_ref['ce'] <= CE_THRESH
    cm_results[('ref', '1t')] = cm_metrics(eval_ref['ue'] < REF_UE1_THRESH, ce_real)
    if USE_UASTHN2:
        cm_results[('ref', '2t')] = cm_metrics(
            compute_final_accept_2t(eval_ref, REF_UE_LOW, REF_UE_HIGH, REF_UE2_MID), ce_real)

if cm_results:
    print("\n" + "="*150)
    print("UNCERTAINTY EVALUATION VS SPATIAL ERROR (CE) — aggregated")
    print(f"  True Positive  = model accepts  AND  CE ≤ {CE_THRESH}")
    print(f"  True Negative  = model rejects  AND  CE > {CE_THRESH}")
    if USE_UASTHN2:
        print(f"  {NAME_2T} decisions use ue_fin for suspicious rows, ue otherwise")
    print(f"PRED ({PRED_NAME}), REF ({REF_NAME})")
    print("="*150)

    agg_headers = ["Source", "Method", "TP", "TN", "FP", "FN", "Total",
                   "Accuracy%", "Acc.Prec%", "Acc.Rec%", "Acc.F1%",
                   "Rej.Prec%", "Rej.Rec%", "Rej.F1%"]
    agg_rows = []
    for (src, scheme), m in cm_results.items():
        sname = NAME_1T if scheme == '1t' else NAME_2T
        agg_rows.append([
            f"{src.upper()}", sname,
            str(m['tp']), str(m['tn']), str(m['fp']), str(m['fn']), str(m['total']),
            f"{m['accuracy']:.2f}", f"{m['pos_precision']:.2f}", f"{m['pos_recall']:.2f}", f"{m['f1_pos']:.2f}",
            f"{m['neg_precision']:.2f}", f"{m['neg_recall']:.2f}", f"{m['f1_neg']:.2f}",
        ])

    cw = [max(len(h), max(len(r[i]) for r in agg_rows)) for i, h in enumerate(agg_headers)]
    hl = " | ".join(h.ljust(cw[i]) for i, h in enumerate(agg_headers))
    print(hl)
    print("-" * len(hl))
    for row in agg_rows:
        print(" | ".join(cell.ljust(cw[i]) for i, cell in enumerate(row)))
    print("="*150 + "\n")


# ==============================================================================
# sss COMPARISON: UASTHN-1 vs UASTHN-2 — classification metric bar chart
#   (shown BEFORE confusion matrices)
# ==============================================================================
compare_sources = []
if ('pred', '1t') in cm_results:
    compare_sources.append(('pred', PRED_NAME))
if USE_REFERENCE and ('ref', '1t') in cm_results:
    compare_sources.append(('ref', REF_NAME))

if compare_sources:
    metrics_keys   = ['accuracy','pos_precision','pos_recall','f1_pos',
                      'neg_precision','neg_recall','f1_neg']
    metrics_labels = ['Accuracy','Acc. Precision','Acc. Recall','Acc. F1',
                      'Rej. Precision','Rej. Recall','Rej. F1']
    palette_1t = ['#5c8be0', '#5ce07a']
    palette_2t = ['#e05c5c', '#e0a05c']

    fig_cmp, ax_cmp = plt.subplots(figsize=(14, 6))
    x = np.arange(len(metrics_labels))
    
    n_s = len(compare_sources)
    n_schemes = 2 if USE_UASTHN2 else 1
    total_bars_per_group = n_s * n_schemes
    bar_w = 0.8 / total_bars_per_group
    
    current_bar_idx = 0
    
    for si, (src, fname) in enumerate(compare_sources):
        m1 = cm_results.get((src, '1t'))
        m2 = cm_results.get((src, '2t'))
        
        # Plot UASTHN-1
        offset_1 = (current_bar_idx - (total_bars_per_group - 1) / 2) * bar_w
        b1 = ax_cmp.bar(x + offset_1, [m1[k] for k in metrics_keys], bar_w,
                         label=f"{NAME_1T} {src.upper()} ({fname})",
                         color=palette_1t[si % 2], alpha=0.85, edgecolor='white')
        
        for bar in b1:
            ax_cmp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                        f"{bar.get_height():.1f}", ha='center', va='bottom', 
                        fontsize=6.5, fontweight='bold', color='#1a3a6b')
        current_bar_idx += 1
        
        # Plot UASTHN-2 and Deltas if enabled
        if USE_UASTHN2 and m2:
            offset_2 = (current_bar_idx - (total_bars_per_group - 1) / 2) * bar_w
            b2 = ax_cmp.bar(x + offset_2, [m2[k] for k in metrics_keys], bar_w,
                             label=f"{NAME_2T} {src.upper()} ({fname})",
                             color=palette_2t[si % 2], alpha=0.85, edgecolor='white')
            for bar in b2:
                ax_cmp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                            f"{bar.get_height():.1f}", ha='center', va='bottom', 
                            fontsize=6.5, fontweight='bold', color='#6b1a1a')
            
            # Draw the delta % text between the two bars
            for i in range(len(metrics_keys)):
                v1, v2 = m1[metrics_keys[i]], m2[metrics_keys[i]]
                delta = v2 - v1
                sign  = '+' if delta >= 0 else ''
                color = 'green' if delta > 0.05 else ('red' if delta < -0.05 else 'gray')
                mid_offset = (offset_1 + offset_2) / 2
                ax_cmp.text(x[i] + mid_offset, max(v1, v2) + 3, f"{sign}{delta:.1f}%",
                            ha='center', va='bottom', fontsize=7.5, color=color, fontweight='bold')
            current_bar_idx += 1

    ax_cmp.set_xticks(x)
    ax_cmp.set_xticklabels(metrics_labels, fontsize=10)
    ax_cmp.set_ylabel("Score (%)", fontsize=11)
    ax_cmp.set_ylim(0, 120)
    
    title_str = f"{NAME_1T} vs {NAME_2T}" if USE_UASTHN2 else f"{NAME_1T} Performance"
    ax_cmp.set_title(f"{title_str}  —  Classification Metric Comparison", fontsize=13, fontweight='bold')
    ax_cmp.legend(fontsize=9, loc='upper left', ncol=2)
    ax_cmp.grid(axis='y', alpha=0.3)
    ax_cmp.axhline(100, color='gray', linestyle=':', linewidth=1, alpha=0.5)
    fig_cmp.tight_layout()
    plt.show()


# ==============================================================================
# sss SCHEME IMPROVEMENT SUMMARY (printed) — UASTHN-1 -> UASTHN-2
# ==============================================================================
if compare_sources and USE_UASTHN2:
    metrics_keys   = ['accuracy','pos_precision','pos_recall','f1_pos',
                      'neg_precision','neg_recall','f1_neg']
    metrics_labels = ['Accuracy','Acc. Precision','Acc. Recall','Acc. F1',
                      'Rej. Precision','Rej. Recall','Rej. F1']
    for src, fname in compare_sources:
        m1 = cm_results.get((src, '1t'))
        m2 = cm_results.get((src, '2t'))
        if not m1 or not m2:
            continue
            
        print(f"\n{'='*80}")
        print(f"SCHEME IMPROVEMENT SUMMARY  —  {src.upper()} ({fname})")
        print(f"  {NAME_1T} → {NAME_2T}")
        print("="*80)
        for key, label in zip(metrics_keys, metrics_labels):
            v1, v2 = m1[key], m2[key]
            delta  = v2 - v1
            sign   = '+' if delta >= 0 else ''
            arrow  = '↑' if delta > 0.05 else ('↓' if delta < -0.05 else '→')
            print(f"  {label:<22}: {NAME_1T}={v1:6.2f}%   {NAME_2T}={v2:6.2f}%   Δ = {sign}{delta:.2f}%  {arrow}")
        print("="*80)


# ==============================================================================
# sss CONFUSION MATRICES
#   Rows = [UASTHN-1, UASTHN-2 (if enabled)]   Cols = [PRED, REF]
# ==============================================================================
def draw_cm_on_ax(ax, m, title):
    total  = m['total']
    cm_arr = np.array([[m['tp'], m['fp']], [m['fn'], m['tn']]])
    ax.matshow(cm_arr, cmap=plt.cm.Greens, alpha=0.55)
    ax.set_xticklabels(['', 'Accurate\n(CE ≤ thresh)', 'Inaccurate\n(CE > thresh)'])
    ax.set_yticklabels(['', 'Accepted', 'Rejected'])
    ax.xaxis.set_ticks_position('bottom')
    ax.set_ylabel('Model Decision', fontsize=10, fontweight='bold', labelpad=8)
    ax.set_xlabel('Actual Spatial Accuracy', fontsize=10, fontweight='bold', labelpad=8)
    ax.set_title(title, fontsize=10, fontweight='bold', pad=12)
    grid_desc = [
        [f"TP\n{m['tp']}\n({100*m['tp']/total:.1f}%)", f"FP\n{m['fp']}\n({100*m['fp']/total:.1f}%)"],
        [f"FN\n{m['fn']}\n({100*m['fn']/total:.1f}%)", f"TN\n{m['tn']}\n({100*m['tn']/total:.1f}%)"],
    ]
    for r in range(2):
        for c in range(2):
            ax.text(c, r, grid_desc[r][c], ha='center', va='center',
                    fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.55))

has_pred_cm = ('pred', '1t') in cm_results
has_ref_cm  = ('ref',  '1t') in cm_results and USE_REFERENCE
n_cm_cols   = (1 if has_pred_cm else 0) + (1 if has_ref_cm else 0)
n_cm_rows   = 2 if USE_UASTHN2 else 1

if n_cm_cols > 0:
    fig_cm, axes_cm = plt.subplots(n_cm_rows, n_cm_cols, figsize=(6.5 * n_cm_cols, 5.5 * n_cm_rows), squeeze=False)
    col_pred = 0
    col_ref  = 1 if (has_pred_cm and has_ref_cm) else 0

    scheme_rows = [('1t', NAME_1T)]
    if USE_UASTHN2:
        scheme_rows.append(('2t', NAME_2T))

    for row_i, (sk, label) in enumerate(scheme_rows):
        if has_pred_cm and ('pred', sk) in cm_results:
            draw_cm_on_ax(axes_cm[row_i][col_pred], cm_results[('pred', sk)],
                          f"{label} — PRED ({PRED_NAME})")
        else:
            axes_cm[row_i][col_pred].axis('off')
        if has_ref_cm and ('ref', sk) in cm_results:
            draw_cm_on_ax(axes_cm[row_i][col_ref], cm_results[('ref', sk)],
                          f"{label} — REF ({REF_NAME})")
        elif n_cm_cols == 2:
            axes_cm[row_i][col_ref].axis('off')

    title_extra = f"  |  {NAME_1T} vs {NAME_2T}" if USE_UASTHN2 else f"  |  {NAME_1T} only"
    fig_cm.suptitle(f"Confusion Matrices{title_extra}  |  CE Threshold = {CE_THRESH} px",
                    fontsize=13, fontweight='bold')
    fig_cm.tight_layout()
    plt.show()


# ==============================================================================
# sss CONFUSION BREAKDOWN — TP/TN/FP/FN side-by-side (bar widths shown in legend)
# ==============================================================================
if compare_sources:
    fig_bd, ax_bd = plt.subplots(figsize=(13, 5))
    cats      = ['TP', 'TN', 'FP', 'FN']
    n_schemes = 2 if USE_UASTHN2 else 1
    n_combo   = len(compare_sources) * n_schemes
    bw        = 0.8 / n_combo
    x_cats    = np.arange(len(cats))
    combo_idx = 0

    palette_1t = ['#5c8be0', '#5ce07a']
    palette_2t = ['#e05c5c', '#e0a05c']

    # Find global max for y-axis scaling safely
    max_val = 0

    for si, (src, fname) in enumerate(compare_sources):
        m1 = cm_results.get((src, '1t'))
        m2 = cm_results.get((src, '2t'))
        
        schemes_to_plot = [(m1, NAME_1T, palette_1t)]
        if USE_UASTHN2 and m2:
            schemes_to_plot.append((m2, NAME_2T, palette_2t))
            
        for m, scheme_name, palette in schemes_to_plot:
            if not m: continue
            
            offset    = (combo_idx - (n_combo - 1) / 2) * bw
            vals_cats = [m['tp'], m['tn'], m['fp'], m['fn']]
            total_v   = m['total']
            
            # Update max_val for y-axis
            max_val = max(max_val, max(vals_cats))
            
            bars = ax_bd.bar(x_cats + offset, vals_cats, bw,
                             color=palette[si % 2], alpha=0.85, edgecolor='white',
                             label=f"{scheme_name} {src.upper()} ({fname})")
            for bar, v in zip(bars, vals_cats):
                ax_bd.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                           f"{v}\n({100*v/total_v:.1f}%)",
                           ha='center', va='bottom', fontsize=7, fontweight='bold')
            combo_idx += 1

    ax_bd.set_xticks(x_cats)
    ax_bd.set_xticklabels(cats, fontsize=13, fontweight='bold')
    ax_bd.set_ylabel("Count", fontsize=11)
    
    title_str = f"{NAME_1T} vs {NAME_2T}" if USE_UASTHN2 else f"{NAME_1T} Performance"
    ax_bd.set_title(f"Confusion Breakdown  |  {title_str}", fontsize=13, fontweight='bold')
    ax_bd.legend(fontsize=7.5, loc='upper right', ncol=2)
    ax_bd.grid(axis='y', alpha=0.3)
    ax_bd.set_ylim(0, max_val * 1.3)
    fig_bd.tight_layout()
    plt.show()


# ==============================================================================
# sss SUSPICIOUS-DATA EFFECTIVENESS — does UE_fin improve decisions?
#   For rows in the suspicious zone (by UE), compare the decision made using
#   raw UE (acc if UE<UE_LOW, i.e. essentially always "rejected"-leaning since
#   UE>=UE_LOW in this zone -> we treat "would-be" decision as UE<ue2_mid using
#   the ORIGINAL ue) vs the decision made using UE_fin (UE_fin<ue2_mid).
#   We then check whether each change (acc->rej or rej->acc) was "correct"
#   relative to ground truth (CE<=thresh).
# ==============================================================================
def analyze_suspicious_effectiveness(src_label, src_df, ce_col, ue_low, ue_high, ue2_mid, has_ue_):
    if not has_ue_ or not USE_UASTHN2:
        return None

    ce_all = dist_df[ce_col].values
    valid  = ~np.isnan(ce_all)
    ce_v   = ce_all[valid]
    ue_v   = src_df['ue'].values[:len(dist_df)][valid]
    ue_fin_v = src_df['ue_fin'].values[:len(dist_df)][valid]

    sus_mask = (ue_v >= ue_low) & (ue_v <= ue_high)
    n_sus = int(np.sum(sus_mask))
    if n_sus == 0:
        return None

    sus_ce, sus_ue, sus_uefin = ce_v[sus_mask], ue_v[sus_mask], ue_fin_v[sus_mask]
    sus_true = sus_ce <= CE_THRESH  # ground-truth correctness

    # "before" decision: compare raw ue against ue2_mid (the only sensible
    # baseline decision available before computing ue2)
    before_acc = sus_ue < ue2_mid
    after_acc  = sus_uefin < ue2_mid

    changed         = before_acc != after_acc
    acc_to_rej      = changed & before_acc & ~after_acc
    rej_to_acc      = changed & ~before_acc & after_acc
    unchanged       = ~changed

    n_changed   = int(np.sum(changed))
    n_acc2rej   = int(np.sum(acc_to_rej))
    n_rej2acc   = int(np.sum(rej_to_acc))
    n_unchanged = int(np.sum(unchanged))

    # correctness of "before" and "after" decisions vs ground truth
    before_correct = before_acc == sus_true
    after_correct  = after_acc  == sus_true

    n_before_correct = int(np.sum(before_correct))
    n_after_correct  = int(np.sum(after_correct))

    # for changed rows specifically: was the change beneficial?
    changed_improved = changed & after_correct & ~before_correct
    changed_worsened = changed & before_correct & ~after_correct
    changed_neutral  = changed & (before_correct == after_correct)

    n_improved = int(np.sum(changed_improved))
    n_worsened = int(np.sum(changed_worsened))
    n_neutral  = int(np.sum(changed_neutral))

    # magnitude of ue change
    ue_delta = sus_uefin - sus_ue
    delta_stats = series_stats(ue_delta)
    delta_abs_stats = series_stats(np.abs(ue_delta))

    return dict(
        src_label=src_label, n_sus=n_sus,
        n_changed=n_changed, n_acc2rej=n_acc2rej, n_rej2acc=n_rej2acc, n_unchanged=n_unchanged,
        n_before_correct=n_before_correct, n_after_correct=n_after_correct,
        n_improved=n_improved, n_worsened=n_worsened, n_neutral=n_neutral,
        delta_stats=delta_stats, delta_abs_stats=delta_abs_stats,
    )


sus_eff_results = []
if pred_has_ue:
    r = analyze_suspicious_effectiveness(f"PRED ({PRED_NAME})", pred_df, 'pred_ce',
                                          PRED_UE_LOW, PRED_UE_HIGH, PRED_UE2_MID, pred_has_ue)
    if r: sus_eff_results.append(r)
if USE_REFERENCE and ref_has_ue:
    r = analyze_suspicious_effectiveness(f"REF ({REF_NAME})", ref_df, 'ref_ce',
                                          REF_UE_LOW, REF_UE_HIGH, REF_UE2_MID, ref_has_ue)
    if r: sus_eff_results.append(r)

if sus_eff_results:
    print("\n" + "="*110)
    print(f"SUSPICIOUS-DATA EFFECTIVENESS ANALYSIS — does UE_fin improve decisions vs raw UE?")
    print(f"  'Before' decision = UE < UE2_MID  |  'After' decision = UE_fin < UE2_MID")
    print(f"  'Correct' = decision matches ground truth (CE<=thresh means should Accept)")
    print("="*110)
    for r in sus_eff_results:
        n = r['n_sus']
        print(f"\n--- {r['src_label']} ---  (n_suspicious = {n})")
        print(f"  Decision changes:")
        print(f"    Unchanged          : {r['n_unchanged']:4d} ({100*r['n_unchanged']/n:.1f}%)")
        print(f"    Accepted -> Rejected: {r['n_acc2rej']:4d} ({100*r['n_acc2rej']/n:.1f}%)")
        print(f"    Rejected -> Accepted: {r['n_rej2acc']:4d} ({100*r['n_rej2acc']/n:.1f}%)")
        print(f"    Total changed       : {r['n_changed']:4d} ({100*r['n_changed']/n:.1f}%)")
        print(f"  Decision correctness:")
        print(f"    Correct BEFORE (raw UE)  : {r['n_before_correct']:4d} ({100*r['n_before_correct']/n:.1f}%)")
        print(f"    Correct AFTER  (UE_fin)  : {r['n_after_correct']:4d} ({100*r['n_after_correct']/n:.1f}%)")
        print(f"    Net improvement          : {r['n_after_correct'] - r['n_before_correct']:+d} rows"
              f"  ({100*(r['n_after_correct'] - r['n_before_correct'])/n:+.1f}%)")
        print(f"  Among changed rows:")
        print(f"    Improved (became correct)  : {r['n_improved']:4d}")
        print(f"    Worsened (became incorrect): {r['n_worsened']:4d}")
        print(f"    Neutral  (still right/wrong): {r['n_neutral']:4d}")
        ds, das = r['delta_stats'], r['delta_abs_stats']
        print(f"  UE_fin - UE  (signed)  : mean={fmt_stat(ds['mean'])}  median={fmt_stat(ds['median'])}  std={fmt_stat(ds['std'])}")
        print(f"  |UE_fin - UE| (magnitude): mean={fmt_stat(das['mean'])}  median={fmt_stat(das['median'])}"
              f"  max={fmt_stat(das['max'])}")
    print("="*110 + "\n")

    # ── Bar chart: changes breakdown ──
    fig_sus, axes_sus = plt.subplots(1, len(sus_eff_results), figsize=(6.5*len(sus_eff_results), 5), squeeze=False)
    for i, r in enumerate(sus_eff_results):
        ax = axes_sus[0][i]
        cats   = ['Unchanged', 'Acc→Rej', 'Rej→Acc']
        vals   = [r['n_unchanged'], r['n_acc2rej'], r['n_rej2acc']]
        colors = ['#9e9e9e', '#f44336', '#4caf50']
        bars = ax.bar(cats, vals, color=colors, alpha=0.85, edgecolor='white',
                       label=[f"width={0.8/len(cats):.2f}"]*len(cats))
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f"{v}\n({100*v/r['n_sus']:.1f}%)",
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        ax.set_title(f"{r['src_label']}\nDecision changes (n={r['n_sus']})", fontsize=10, fontweight='bold')
        ax.set_ylabel("Count")
        ax.grid(axis='y', alpha=0.3)

        # Secondary annotation: improved/worsened/neutral among changed
        text = (f"Of changed rows:\n"
                f"  Improved: {r['n_improved']}\n"
                f"  Worsened: {r['n_worsened']}\n"
                f"  Neutral : {r['n_neutral']}")
        ax.text(0.98, 0.97, text, transform=ax.transAxes, fontsize=8.5,
                va='top', ha='right',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.88))
    fig_sus.suptitle("Suspicious-Data Effectiveness: Decision Changes (UE → UE_fin)", fontsize=13, fontweight='bold')
    fig_sus.tight_layout()
    plt.show()

# ==============================================================================
# sss DIST/DISP vs UE/UE_FIN — heatmap analysis
#   Rows: dist-vs-ue | dist-vs-ue_fin | disp-vs-ue | disp-vs-ue_fin
#   Cols: 1=PRED(true/false) 2=PRED(frequency) 3=REF(true/false) 4=REF(frequency)
#   dist = |iter0 -> iter6| straight-line distance
#   disp = total path length iter0->iter1->...->iter6
# ==============================================================================
if HAS_ITER_PATH and (pred_has_ue):

    def build_heatmap_data(src_df, ce_col, dist_arr, disp_arr, ue_col):
        ce_all = dist_df[ce_col].values
        valid  = (~np.isnan(ce_all)) & (~np.isnan(dist_arr)) & (~np.isnan(disp_arr))
        if 'ue' not in src_df.columns:
            return None
        ue_v = src_df[ue_col].values[:len(dist_df)]
        valid = valid & (~np.isnan(ue_v))
        if valid.sum() == 0:
            return None
        return dict(
            ce=ce_all[valid], dist=dist_arr[valid], disp=disp_arr[valid], ue=ue_v[valid],
            true_mask=ce_all[valid] <= CE_THRESH,
        )

    def plot_heatmap_pair(
        ax_tf, ax_freq,
        x_vals, y_vals, true_mask,
        x_label, y_label,
        x_range=None,
        y_range=None,
        n_bins=25
    ):
        """
        x_range = (xmin, xmax)
        y_range = (ymin, ymax)

        Heatmap is computed ONLY inside selected interval.
        """

        # -------------------------------------------------
        # Manual subregion filtering
        # -------------------------------------------------
        mask = np.ones(len(x_vals), dtype=bool)

        if x_range is not None:
            xmin, xmax = x_range
            mask &= (x_vals >= xmin) & (x_vals <= xmax)

        if y_range is not None:
            ymin, ymax = y_range
            mask &= (y_vals >= ymin) & (y_vals <= ymax)

        x_vals = x_vals[mask]
        y_vals = y_vals[mask]
        true_mask = true_mask[mask]

        if len(x_vals) == 0:
            ax_tf.set_title("No data in selected range")
            ax_freq.set_title("No data in selected range")
            return

        # -------------------------------------------------
        # Bin edges
        # -------------------------------------------------
        if x_range is None:
            x_edges = np.linspace(
                x_vals.min(),
                x_vals.max() + 1e-9,
                n_bins + 1
            )
        else:
            x_edges = np.linspace(
                x_range[0],
                x_range[1],
                n_bins + 1
            )

        if y_range is None:
            y_edges = np.linspace(
                y_vals.min(),
                y_vals.max() + 1e-9,
                n_bins + 1
            )
        else:
            y_edges = np.linspace(
                y_range[0],
                y_range[1],
                n_bins + 1
            )

        # -------------------------------------------------
        # Histograms
        # -------------------------------------------------
        freq, _, _ = np.histogram2d(
            x_vals, y_vals,
            bins=[x_edges, y_edges]
        )

        true_counts, _, _ = np.histogram2d(
            x_vals[true_mask],
            y_vals[true_mask],
            bins=[x_edges, y_edges]
        )

        false_counts, _, _ = np.histogram2d(
            x_vals[~true_mask],
            y_vals[~true_mask],
            bins=[x_edges, y_edges]
        )

        total = true_counts + false_counts

        with np.errstate(invalid='ignore', divide='ignore'):
            ratio = np.where(
                total > 0,
                true_counts / total,
                np.nan
            )

        # -------------------------------------------------
        # True/False heatmap
        # -------------------------------------------------
        im1 = ax_tf.imshow(
            ratio.T,
            origin='lower',
            aspect='auto',
            cmap=plt.cm.RdYlGn,
            vmin=0,
            vmax=1,
            extent=[
                x_edges[0], x_edges[-1],
                y_edges[0], y_edges[-1]
            ]
        )

        plt.colorbar(
            im1, ax=ax_tf,
            label='Fraction True',
            fraction=0.046, pad=0.04
        )

        ax_tf.set_xlabel(x_label, fontsize=9)
        ax_tf.set_ylabel(y_label, fontsize=9)

        # -------------------------------------------------
        # Frequency heatmap
        # -------------------------------------------------
        im2 = ax_freq.imshow(
            freq.T,
            origin='lower',
            aspect='auto',
            cmap=plt.cm.Blues,
            extent=[
                x_edges[0], x_edges[-1],
                y_edges[0], y_edges[-1]
            ]
        )

        plt.colorbar(
            im2, ax=ax_freq,
            label='Sample count',
            fraction=0.046, pad=0.04
        )

        ax_freq.set_xlabel(x_label, fontsize=9)
        ax_freq.set_ylabel(y_label, fontsize=9)


    # Build dist/disp arrays for REF too, if iteration cols exist there
    ref_dist_vals = np.full(len(ref_df), np.nan) if ref_df is not None else None
    ref_disp_vals = np.full(len(ref_df), np.nan) if ref_df is not None else None
    ref_iter_cols_present = []
    if ref_df is not None:
        ref_iter_cols_present = [
            s for s in ITER_SUFFIXES
            if all(f"{c}{s}" in ref_df.columns for c in ['x1','y1','x2','y2','x3','y3','x4','y4'])
        ]
        if len(ref_iter_cols_present) > 0:
            for idx, row in ref_df.iterrows():
                sat_path = resolve_img_path(row["sat"]) if "sat" in row.index else None
                if sat_path is None:
                    continue
                try:
                    with Image.open(sat_path) as im:
                        img_w, img_h = im.size
                except Exception:
                    continue
                d, disp = compute_iter_dist_disp(row, img_h, img_w)
                ref_dist_vals[idx] = d
                ref_disp_vals[idx] = disp

    pred_data = build_heatmap_data(pred_df, 'pred_ce', pred_df['_iter_dist'].values, pred_df['_iter_disp'].values, 'ue')
    pred_data_fin = build_heatmap_data(pred_df, 'pred_ce', pred_df['_iter_dist'].values, pred_df['_iter_disp'].values, 'ue_fin') if USE_UASTHN2 else None

    ref_data = None
    ref_data_fin = None
    if USE_REFERENCE and ref_df is not None and ref_has_ue and len(ref_iter_cols_present) > 0:
        ref_df['_iter_dist'] = ref_dist_vals
        ref_df['_iter_disp'] = ref_disp_vals
        ref_data = build_heatmap_data(ref_df, 'ref_ce', ref_dist_vals, ref_disp_vals, 'ue')
        if USE_UASTHN2:
            ref_data_fin = build_heatmap_data(ref_df, 'ref_ce', ref_dist_vals, ref_disp_vals, 'ue_fin')

    # Determine which rows to show
    heatmap_rows = [("dist", "ue", "Dist (iter0→iter6) vs UE", pred_data, ref_data)]
    heatmap_rows.append(("disp", "ue", "Disp (path length) vs UE", pred_data, ref_data))
    if USE_UASTHN2:
        heatmap_rows.append(("dist", "ue_fin", "Dist (iter0→iter6) vs UE_fin", pred_data_fin, ref_data_fin))
        heatmap_rows.append(("disp", "ue_fin", "Disp (path length) vs UE_fin", pred_data_fin, ref_data_fin))

    n_hrows = len(heatmap_rows)
    fig_hm, axes_hm = plt.subplots(n_hrows, 4, figsize=(22, 5 * n_hrows))
    if n_hrows == 1:
        axes_hm = axes_hm.reshape(1, 4)

    for ri, (quant, ue_kind, title, pdat, rdat) in enumerate(heatmap_rows):
        ue_label = "UE" if ue_kind == "ue" else "UE_fin"
        y_label  = "Dist (px)" if quant == "dist" else "Disp (px)"

        # Cols 1,2: PRED
        if pdat is not None:
            y_vals = pdat['dist'] if quant == "dist" else pdat['disp']
            plot_heatmap_pair(axes_hm[ri][0], axes_hm[ri][1],
                              pdat['ue'], y_vals, pdat['true_mask'],
                              f"PRED {ue_label}", y_label)
            axes_hm[ri][0].set_title(f"PRED ({PRED_NAME})\n{title} — True/False", fontsize=9, fontweight='bold')
            axes_hm[ri][1].set_title(f"PRED ({PRED_NAME})\n{title} — Frequency", fontsize=9, fontweight='bold')
        else:
            axes_hm[ri][0].axis('off')
            axes_hm[ri][1].axis('off')

        # Cols 3,4: REF
        if rdat is not None:
            y_vals = rdat['dist'] if quant == "dist" else rdat['disp']
            plot_heatmap_pair(axes_hm[ri][2], axes_hm[ri][3],
                              rdat['ue'], y_vals, rdat['true_mask'],
                              f"REF {ue_label}", y_label)
            axes_hm[ri][2].set_title(f"REF ({REF_NAME})\n{title} — True/False", fontsize=9, fontweight='bold')
            axes_hm[ri][3].set_title(f"REF ({REF_NAME})\n{title} — Frequency", fontsize=9, fontweight='bold')
        else:
            axes_hm[ri][2].axis('off')
            axes_hm[ri][3].axis('off')

    fig_hm.suptitle(
        "Distance/Displacement (iteration path) vs UE / UE_fin — Heatmaps\n"
        "dist = straight-line(iter0→iter6)  |  disp = total path length(iter0→...→iter6)\n"
        "Cols 1&3: green=True(CE≤thresh) / red=False(CE>thresh)  |  Cols 2&4: sample frequency (white→blue)",
        fontsize=12, fontweight='bold', y=1)
    fig_hm.tight_layout()
    plt.show()


    # --- SUB-HEATMAP---

    ref_data = None
    ref_data_fin = None
    if USE_REFERENCE and ref_df is not None and ref_has_ue and len(ref_iter_cols_present) > 0:
        ref_df['_iter_dist'] = ref_dist_vals
        ref_df['_iter_disp'] = ref_disp_vals
        ref_data = build_heatmap_data(ref_df, 'ref_ce', ref_dist_vals, ref_disp_vals, 'ue')
        if USE_UASTHN2:
            ref_data_fin = build_heatmap_data(ref_df, 'ref_ce', ref_dist_vals, ref_disp_vals, 'ue_fin')

    # Determine which rows to show
    heatmap_rows = [("dist", "ue", "Dist (iter0→iter6) vs UE", pred_data, ref_data)]
    heatmap_rows.append(("disp", "ue", "Disp (path length) vs UE", pred_data, ref_data))
    if USE_UASTHN2:
        heatmap_rows.append(("dist", "ue_fin", "Dist (iter0→iter6) vs UE_fin", pred_data_fin, ref_data_fin))
        heatmap_rows.append(("disp", "ue_fin", "Disp (path length) vs UE_fin", pred_data_fin, ref_data_fin))

    n_hrows = len(heatmap_rows)
    fig_hm, axes_hm = plt.subplots(n_hrows, 4, figsize=(22, 5 * n_hrows))
    if n_hrows == 1:
        axes_hm = axes_hm.reshape(1, 4)

    for ri, (quant, ue_kind, title, pdat, rdat) in enumerate(heatmap_rows):
        ue_label = "UE" if ue_kind == "ue" else "UE_fin"
        y_label  = "Dist (px)" if quant == "dist" else "Disp (px)"

        # Cols 1,2: PRED
        if pdat is not None:
            y_vals = pdat['dist'] if quant == "dist" else pdat['disp']
            y_range = DIST_RANGE if quant == "dist" else DISP_RANGE
            plot_heatmap_pair(axes_hm[ri][0], axes_hm[ri][1],
                              pdat['ue'], y_vals, pdat['true_mask'],
                              f"PRED {ue_label}", y_label, x_range=UE_RANGE, y_range=y_range, n_bins=HEAT_BINS)
            axes_hm[ri][0].set_title(f"PRED ({PRED_NAME})\n{title} — True/False", fontsize=9, fontweight='bold')
            axes_hm[ri][1].set_title(f"PRED ({PRED_NAME})\n{title} — Frequency", fontsize=9, fontweight='bold')
        else:
            axes_hm[ri][0].axis('off')
            axes_hm[ri][1].axis('off')

        # Cols 3,4: REF
        if rdat is not None:
            y_vals = rdat['dist'] if quant == "dist" else rdat['disp']
            y_range = DIST_RANGE if quant == "dist" else DISP_RANGE
            plot_heatmap_pair(axes_hm[ri][2], axes_hm[ri][3],
                              rdat['ue'], y_vals, rdat['true_mask'],
                              f"REF {ue_label}", y_label, x_range=UE_RANGE, y_range=y_range, n_bins=HEAT_BINS)
            axes_hm[ri][2].set_title(f"REF ({REF_NAME})\n{title} — True/False", fontsize=9, fontweight='bold')
            axes_hm[ri][3].set_title(f"REF ({REF_NAME})\n{title} — Frequency", fontsize=9, fontweight='bold')
        else:
            axes_hm[ri][2].axis('off')
            axes_hm[ri][3].axis('off')

    fig_hm.suptitle(
        "Distance/Displacement (iteration path) vs UE / UE_fin — Heatmaps\n"
        "dist = straight-line(iter0→iter6)  |  disp = total path length(iter0→...→iter6)\n"
        "Cols 1&3: green=True(CE≤thresh) / red=False(CE>thresh)  |  Cols 2&4: sample frequency (white→blue)",
        fontsize=12, fontweight='bold', y=1)
    fig_hm.tight_layout()
    plt.show()
else:
    print("\n[INFO] Dist/Disp vs UE heatmap section skipped: no iteration path columns or no UE data.\n")

# ==============================================================================
# sss FP REDUCTION ANALYSIS VIA DIST / DISP THRESHOLDS
# ==============================================================================
if HAS_ITER_PATH and pred_has_ue:
    
    # ── 1. Define the Thresholds (Change these values as needed) ──────────────
    DIST_TH = 110  
    DISP_TH = 110 
    
    # Extract valid data arrays
    ce_v   = dist_df['pred_ce'].values
    dist_v = pred_df['_iter_dist'].values
    disp_v = pred_df['_iter_disp'].values
    
    valid = (~np.isnan(ce_v)) & (~np.isnan(dist_v)) & (~np.isnan(disp_v))
    ce_ok = ce_v <= CE_THRESH
    
    # ── 2. Identify TP and FP masks for UASTHN-1 and UASTHN-2 ─────────────────
    acc_1t     = (pred_df['ue'].values < PRED_UE1_THRESH) & valid
    fp_1t_mask = acc_1t & ~ce_ok
    tp_1t_mask = acc_1t & ce_ok
    
    if USE_UASTHN2:
        acc_2t     = compute_final_accept_2t(pred_df, PRED_UE_LOW, PRED_UE_HIGH, PRED_UE2_MID).values & valid
        fp_2t_mask = acc_2t & ~ce_ok
        tp_2t_mask = acc_2t & ce_ok

    # Helper to calculate how TP/FP changes with the new condition
    def calc_filter_stats(tp_mask, fp_mask, filter_mask):
        orig_tp = tp_mask.sum()
        orig_fp = fp_mask.sum()
        new_tp  = (tp_mask & filter_mask).sum()
        new_fp  = (fp_mask & filter_mask).sum()
        
        tp_lessened = orig_tp - new_tp  # (We want this to be 0)
        fp_lessened = orig_fp - new_fp  # (We want this to be high)
        return orig_tp, orig_fp, new_tp, new_fp, tp_lessened, fp_lessened

    # Keep predictions where Dist/Disp are GREATER THAN OR EQUAL to the threshold
    dist_filter = dist_v >= DIST_TH
    disp_filter = disp_v >= DISP_TH
    
    # ── 3. Build and Print the Summary Table ──────────────────────────────────
    stats_rows = []
    
    # UASTHN-1 stats
    o_tp, o_fp, n_tp, n_fp, l_tp, r_fp = calc_filter_stats(tp_1t_mask, fp_1t_mask, dist_filter)
    stats_rows.append([NAME_1T, f"Dist >= {DIST_TH}", o_tp, o_fp, n_tp, n_fp, l_tp, r_fp])
    
    o_tp, o_fp, n_tp, n_fp, l_tp, r_fp = calc_filter_stats(tp_1t_mask, fp_1t_mask, disp_filter)
    stats_rows.append([NAME_1T, f"Disp >= {DISP_TH}", o_tp, o_fp, n_tp, n_fp, l_tp, r_fp])
    
    # UASTHN-2 stats
    if USE_UASTHN2:
        o_tp, o_fp, n_tp, n_fp, l_tp, r_fp = calc_filter_stats(tp_2t_mask, fp_2t_mask, dist_filter)
        stats_rows.append([NAME_2T, f"Dist >= {DIST_TH}", o_tp, o_fp, n_tp, n_fp, l_tp, r_fp])
        
        o_tp, o_fp, n_tp, n_fp, l_tp, r_fp = calc_filter_stats(tp_2t_mask, fp_2t_mask, disp_filter)
        stats_rows.append([NAME_2T, f"Disp >= {DISP_TH}", o_tp, o_fp, n_tp, n_fp, l_tp, r_fp])

    print("\n" + "="*120)
    print("FALSE POSITIVE REDUCTION VIA DIST & DISP THRESHOLDS")
    print(f"Condition: Prediction is ACCEPTED only if (UE Condition OK) AND (Dist/Disp >= Thresh)")
    print(f"Settings: DIST_TH = {DIST_TH}, DISP_TH = {DISP_TH}")
    print("="*120)
    
    headers = ["Scheme", "Condition", "Orig TP", "Orig FP", "New TP", "New FP", "TP Lost (< Thresh)", "FP Lessened (< Thresh)"]
    cw = [max(len(str(x)) for x in col) for col in zip(headers, *stats_rows)]
    row_format = " | ".join([f"{{:<{w}}}" for w in cw])
    
    print(row_format.format(*headers))
    print("-" * (sum(cw) + 3 * (len(cw) - 1)))
    for row in stats_rows:
        print(row_format.format(*row))
    print("="*120 + "\n")


    # ── 4. Plot 4 Rows for TP vs FP Data (Twin Axes) ──────────────────────────
    # ── 4. Plot 4 Rows for TP vs FP Data (Twin Axes) ──────────────────────────
    fig_fp, axes_fp = plt.subplots(4 if USE_UASTHN2 else 2, 1, figsize=(10, 18 if USE_UASTHN2 else 9))
    
    def plot_twin_tp_fp(ax_left, tp_data, fp_data, title, xlabel, thresh, bin_count=35, log=True):
        ax_right = ax_left.twinx()
        
        # Determine common bins
        all_vals = np.concatenate([tp_data, fp_data]) if (len(tp_data) + len(fp_data)) > 0 else np.array([0, 1])
        bins = np.linspace(all_vals.min(), max(all_vals.max(), all_vals.min() + 1), bin_count)
        
        # Plot Histograms
        if len(tp_data) > 0:
            ax_left.hist(tp_data, bins=bins, color='#4caf50', alpha=0.6, edgecolor='white', label=f'TP (n={len(tp_data)})')
        if len(fp_data) > 0:
            ax_right.hist(fp_data, bins=bins, color='#f44336', alpha=0.5, edgecolor='white', label=f'FP (n={len(fp_data)})')
            
        # Draw threshold line
        ax_left.axvline(thresh, color='black', linestyle='--', linewidth=2, label=f'Thresh = {thresh}')
        
        # Labels and Scales
        ax_left.set_ylabel(f"Count{' (log)' if log else ''}: TP", color='#2e7d32', fontsize=10)
        ax_right.set_ylabel(f"Count{' (log)' if False else ''}: FP", color='#c62828', fontsize=10)
        
        if log:
            ax_left.set_yscale('log')
            # ax_right.set_yscale('log')
            
        ax_left.tick_params(axis='y', labelcolor='#2e7d32')
        ax_right.tick_params(axis='y', labelcolor='#c62828')
        ax_left.set_xlabel(xlabel, fontsize=10)
        ax_left.set_title(title, fontsize=11, fontweight='bold')
        
        # Combine legends
        handles1, labels1 = ax_left.get_legend_handles_labels()
        handles2, labels2 = ax_right.get_legend_handles_labels()
        ax_left.legend(handles1 + handles2, labels1 + labels2, fontsize=9, loc='upper right')
        
        # Annotate lessened stats (values strictly LESS THAN thresh are removed)
        tp_removed = (tp_data < thresh).sum()
        fp_removed = (fp_data < thresh).sum()
        ax_left.text(0.02, 0.95, f"TP Lost (<{thresh}): {tp_removed}\nFP Lessened (<{thresh}): {fp_removed}", 
                transform=ax_left.transAxes, ha='left', va='top', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9, edgecolor='gray'))
        ax_left.grid(axis='y', alpha=0.3, color='#2e7d32')
        ax_right.grid(axis='y', alpha=0.3, color='#c62828')

    # Row 1 & 2: UASTHN-1
    plot_twin_tp_fp(axes_fp[0], dist_v[tp_1t_mask], dist_v[fp_1t_mask], 
                    f"{NAME_1T} TP vs FP — Distance Distribution", "Distance (px)", DIST_TH)
    
    plot_twin_tp_fp(axes_fp[1], disp_v[tp_1t_mask], disp_v[fp_1t_mask], 
                    f"{NAME_1T} TP vs FP — Displacement Distribution", "Displacement (px)", DISP_TH)
    
    # Row 3 & 4: UASTHN-2
    if USE_UASTHN2:
        plot_twin_tp_fp(axes_fp[2], dist_v[tp_2t_mask], dist_v[fp_2t_mask], 
                        f"{NAME_2T} TP vs FP — Distance Distribution", "Distance (px)", DIST_TH)
        
        plot_twin_tp_fp(axes_fp[3], disp_v[tp_2t_mask], disp_v[fp_2t_mask], 
                        f"{NAME_2T} TP vs FP — Displacement Distribution", "Displacement (px)", DISP_TH)
        
    fig_fp.suptitle("True Positives vs False Positives (Dist & Disp)\nEvaluating auxiliary threshold rejections", 
                    fontsize=14, fontweight='bold', y=0.99)
    fig_fp.tight_layout(h_pad=3.0)
    plt.show()
else:
    print("\n[INFO] FP Reduction section skipped: Iteration path or UE columns missing.\n")


# ==============================================================================
# sss RENDER OVERLAYS — FALSE PREDICTIONS ONLY
#   (UASTHN-2 classification uses ue_fin for all decisions)
# ==============================================================================
if pred_has_ue:
    ue_vals_s  = pred_df['ue']
    ue_fin_s   = pred_df['ue_fin']
    if USE_UASTHN2:
        pred_df['_ue_class'] = [
            classify_ue_2t_with_zone(
                ue_vals_s.iloc[i], ue_fin_s.iloc[i],
                PRED_UE_LOW, PRED_UE_HIGH, PRED_UE2_MID)
            for i in range(len(pred_df))
        ]
    else:
        pred_df['_ue_class'] = [
            'accepted' if ue_vals_s.iloc[i] < PRED_UE1_THRESH else 'rejected'
            for i in range(len(pred_df))
        ]
    pred_df['_final_accept'] = pred_df['_ue_class'].isin(['accepted', 'suspicious_acc'])
else:
    pred_df['_ue_class']     = 'accepted'
    pred_df['_final_accept'] = True


def is_false_prediction(idx):
    if 'pred_ce' not in dist_df.columns or idx not in dist_df.index:
        return False
    ce_ok  = dist_df.loc[idx, 'pred_ce'] <= CE_THRESH
    ue_acc = bool(pred_df.loc[idx, '_final_accept'])
    return ce_ok != ue_acc


false_indices = [idx for idx in pred_df.index if is_false_prediction(idx)]
if RANDOM_SELECTION:
    if SEED is not None:
        np.random.seed(SEED)
    false_indices = list(np.random.permutation(false_indices))

results_overlay = []
rendered_count  = 0

for idx in false_indices:
    if NUM_IMAGES is not None and rendered_count >= NUM_IMAGES:
        break

    row_pred = pred_df.loc[idx]
    sat_path = resolve_img_path(row_pred["sat"])
    th_path  = resolve_img_path(row_pred["th"])
    if sat_path is None or th_path is None:
        print(f"[SKIP] Missing image at row {idx}")
        continue

    big_map   = np.array(Image.open(sat_path).convert("RGB"))
    small_map = np.array(Image.open(th_path).convert("RGB"))
    img_h, img_w = big_map.shape[:2]

    h, w    = small_map.shape[:2]
    src_pts = np.array([[0, 0], [w-1, 0], [0, h-1], [w-1, h-1]], dtype=np.float32)
    dst_pts = np.array([[row_pred["x1"], row_pred["y1"]],
                        [row_pred["x2"], row_pred["y2"]],
                        [row_pred["x3"], row_pred["y3"]],
                        [row_pred["x4"], row_pred["y4"]]], dtype=np.float32)
    H, _ = cv2.findHomography(src_pts, dst_pts)
    if H is None:
        print(f"[FAIL] Homography failed at row {idx}")
        continue

    warped  = cv2.warpPerspective(small_map, H, (img_w, img_h))
    overlay = big_map.copy()
    mask    = np.any(warped != 0, axis=2)
    if ALPHA_BLEND >= 1.0:
        overlay[mask] = warped[mask]
    else:
        overlay[mask] = (ALPHA_BLEND * warped[mask] + (1.0 - ALPHA_BLEND) * overlay[mask]).astype(np.uint8)

    if SHOW_GT and gt_df is not None and idx < len(gt_df):
        draw_quad(overlay, gt_df.iloc[idx], color=(0, 255, 0),  thickness=4, label="GT")
    if SHOW_REF and USE_REFERENCE and ref_df is not None and idx < len(ref_df):
        draw_quad(overlay, ref_df.iloc[idx], color=(0, 0, 255), thickness=3, label="Ref")
    if SHOW_PRED:
        draw_quad(overlay, row_pred, color=(255, 0, 0), thickness=2, label="Pred")
    if HAS_ITER_PATH:
        draw_iteration_path(overlay, row_pred, img_h, img_w)

    ue_class = row_pred['_ue_class']
    overlay  = add_border(overlay, ue_class, border_width=28)

    pred_ce   = dist_df.loc[idx, 'pred_ce']   if 'pred_ce'   in dist_df.columns and idx in dist_df.index else None
    pred_mace = dist_df.loc[idx, 'pred_mace'] if 'pred_mace' in dist_df.columns and idx in dist_df.index else None
    ref_ce    = dist_df.loc[idx, 'ref_ce']    if USE_REFERENCE and 'ref_ce'    in dist_df.columns and idx in dist_df.index else None
    ref_mace  = dist_df.loc[idx, 'ref_mace']  if USE_REFERENCE and 'ref_mace'  in dist_df.columns and idx in dist_df.index else None

    pred_ue     = row_pred["ue"]     if "ue"     in row_pred.index and pd.notna(row_pred["ue"])     else None
    pred_ue_fin = row_pred["ue_fin"] if "ue_fin" in row_pred.index and pd.notna(row_pred["ue_fin"]) else None

    title_text = f"{idx}: {os.path.basename(th_path)}"
    if pred_ce is not None:
        ue_s    = f"{pred_ue:.2f}"     if pred_ue     is not None else "—"
        mace_s  = f"{pred_mace:.1f}px" if pred_mace   is not None else "—"
        if USE_UASTHN2:
            uefin_s = f"{pred_ue_fin:.2f}" if pred_ue_fin is not None else "—"
            title_text += f"\nPRED CE:{pred_ce:.1f}px  MACE:{mace_s}  UE:{ue_s}  UE_fin:{uefin_s}"
        else:
            title_text += f"\nPRED CE:{pred_ce:.1f}px  MACE:{mace_s}  UE:{ue_s}"
    if ref_ce is not None:
        ref_ue_val = None
        if ref_df is not None and idx < len(ref_df):
            rv = ref_df.loc[idx]
            if "ue" in rv.index and pd.notna(rv["ue"]):
                ref_ue_val = rv["ue"]
        ue_s       = f"{ref_ue_val:.2f}" if ref_ue_val is not None else "—"
        ref_mace_s = f"{ref_mace:.1f}px" if ref_mace   is not None else "—"
        title_text += f"\nREF  CE:{ref_ce:.1f}px  MACE:{ref_mace_s}  UE:{ue_s}"

    results_overlay.append((overlay, title_text))
    rendered_count += 1


# ==============================================================================
# sss GRID PLOT — false predictions only
# ==============================================================================
if results_overlay:
    n_rows    = math.ceil(len(results_overlay) / GRID_COLS)
    fig_g, axes_g = plt.subplots(n_rows, GRID_COLS, figsize=(5 * GRID_COLS, 5 * n_rows))
    fig_g.subplots_adjust(top=0.92, hspace=0.15, wspace=0.1)
    axes_flat = axes_g.flatten() if n_rows > 1 or GRID_COLS > 1 else np.array([axes_g])

    for i, (img, title) in enumerate(results_overlay):
        axes_flat[i].imshow(img)
        axes_flat[i].set_title(title, fontsize=7.5, color='darkred')
        axes_flat[i].axis("off")
    for i in range(len(results_overlay), len(axes_flat)):
        axes_flat[i].axis("off")

    legend_elements = []
    if SHOW_GT:
        legend_elements.append(Patch(facecolor='green', edgecolor='green',
                                     label=f'GT: {GT_NAME}', linewidth=3))
    if SHOW_REF and USE_REFERENCE:
        legend_elements.append(Patch(facecolor='blue', edgecolor='blue',
                                     label=f'REF: {REF_NAME}', linewidth=3))
    if SHOW_PRED:
        legend_elements.append(Patch(facecolor='red', edgecolor='red',
                                     label=f'PRED: {PRED_NAME}', linewidth=2))

    if USE_UASTHN2:
        legend_elements += [
            Patch(facecolor='lime',   edgecolor='green',
                  label=f'Border: ue_fin<{PRED_UE2_MID} (non-sus: ue<{PRED_UE_LOW}) → Accepted'),
            Patch(facecolor='salmon', edgecolor='red',
                  label=f'Border: ue_fin≥{PRED_UE2_MID} (non-sus: ue>{PRED_UE_HIGH}) → Rejected'),
            Patch(facecolor='yellow', edgecolor='goldenrod',
                  label=f'Border TL yellow = was suspicious ({PRED_UE_LOW}≤ue≤{PRED_UE_HIGH})'),
            Patch(facecolor='none', edgecolor='none',
                  label='BR color shows final decision from ue_fin'),
            Patch(facecolor='none', edgecolor='none',
                  label=f'All shown = FALSE {NAME_2T} predictions'),
        ]
    else:
        legend_elements += [
            Patch(facecolor='lime',   edgecolor='green',
                  label=f'Border: ue<{PRED_UE1_THRESH} → Accepted'),
            Patch(facecolor='salmon', edgecolor='red',
                  label=f'Border: ue≥{PRED_UE1_THRESH} → Rejected'),
            Patch(facecolor='none', edgecolor='none',
                  label=f'All shown = FALSE {NAME_1T} predictions'),
        ]

    if HAS_ITER_PATH:
        legend_elements += [
            Patch(facecolor='white', edgecolor='black',
                  label='● White = image center (step 0)'),
            Patch(facecolor='red',   edgecolor='black',
                  label='● Red = iter steps 1–6  |  Orange = final pred center (F)'),
            Patch(facecolor='none',  edgecolor='none',
                  label='--- Red dashed = prediction path'),
        ]
    legend_elements.append(Patch(facecolor='none', edgecolor='none',
                                  label='CE: Center Error | MACE: Mean Absolute Corner Error'))

    fig_g.legend(handles=legend_elements,
                 loc='upper center', bbox_to_anchor=(0.5, 0.99),
                 ncol=3, fontsize=8.5, frameon=True, fancybox=True, shadow=True)

    path_note = "  |  Path: center→iter1…6→final" if HAS_ITER_PATH else ""
    alg_name  = NAME_2T if USE_UASTHN2 else NAME_1T
    thresh_note = (f"UE_LOW={PRED_UE_LOW}  UE_HIGH={PRED_UE_HIGH}  UE2_MID={PRED_UE2_MID}"
                    if USE_UASTHN2 else f"UE1_THRESH={PRED_UE1_THRESH}")
    fig_g.suptitle(
        f"False Predictions Grid ({alg_name})  |  "
        f"{thresh_note}  CE={CE_THRESH}px{path_note}",
        fontsize=11, fontweight='bold', y=1.02)

    if SAVE_FIGURE:
        out_path = SAVE_PATH if os.path.isabs(SAVE_PATH) else os.path.join(PROJECT_ROOT, SAVE_PATH)
        plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")

    plt.show()
else:
    print("\n[INFO] No false predictions found — nothing to plot in the overlay grid.\n")